In [10]:
number_of_customers = 1000
starting_customer_index = 1000
number_of_addresses = 1500
starting_address_index = 5000
number_of_products = 1000
starting_product_index = 1000
number_of_orders = 2500
starting_order_index = 100
number_of_order_items = 5000
starting_order_item_index = 100
number_of_reviews = 3000
starting_review_index = 100
number_of_categories = 200
starting_category_index = 500
number_of_wishlists = 2000
starting_wishlist_index = 10
number_of_payments = 3500
starting_payment_index = 200
number_of_campaigns = 500
starting_campaign_index = 1
number_of_sessions = 5000
starting_session_index = 100
number_of_suppliers = 1000
starting_supplier_index = 10000
number_of_inventories = 1500
starting_inventory_index = 10000
number_of_carts = 3000
starting_cart_index = 1000

In [11]:
from collections import defaultdict

# Global data structures
product_prices = {}  # product_num -> price
product_cost_prices = {}  # product_num -> cost
product_tax_rate = 0.10  # Flat 10% tax rate
order_items_data = defaultdict(list)  # order_id -> list of {discount, tax, line_total}
order_coupons = {}  # order_id -> coupon code (NEW)
order_discount_rates = {}  # order_id -> discount rate (NEW)

In [12]:
import re

def extract_product_number(prod_id: str) -> str:
    print(f"Extracting product number from '{prod_id}'")
    
    # Order prefixes from longest to shortest
    cleaned = re.sub(r'^(PROD_|P_|P|ORD_|O_|O)', '', prod_id, flags=re.IGNORECASE)
    
    match = re.match(r'(\d+)', cleaned)
    if match:
        return int(match.group(1))
    else:
        raise ValueError(f"No numeric product ID found in '{prod_id}'")
    # return prod_id


### Customer Table Generator

In [13]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, timezone
import random
import string
import re

fake = Faker(["en_US", "en_GB", "en_CA", "en_AU"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

GENDER_MAP = {
    "M": "Male", "m": "Male", "male": "Male", "man": "Male",
    "F": "Female", "f": "Female", "female": "Female", "woman": "Female",
    "O": "Other", "o": "Other", "non-binary": "Other", "nb": "Other",
    "1": "Male", "2": "Female",
}
GENDER_CANONICAL = ["Male", "Female", "Other", "Prefer not to say"]
STATUS_CANONICAL = ["Active", "Inactive", "Blocked", "Pending", "Closed"]
DUMMY_EMAIL_DOMAINS = ["example.com", "test.com", "mailinator.com", "tempmail.com"]

def generate_user_id(index):
    """Generate numeric integer user_id (> 0)."""
    return index

def to_iso8601(dt):
    """Convert to ISO-8601 with UTC timezone."""
    if dt is None:
        return None
    if isinstance(dt, str):
        try:
            parsed = pd.to_datetime(dt, errors="coerce")
            if pd.isna(parsed):
                return None
            return parsed.strftime("%Y-%m-%dT%H:%M:%SZ")
        except:
            return None
    if isinstance(dt, (datetime, pd.Timestamp)):
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return None

def to_iso8601_date(dt):
    """Convert to YYYY-MM-DD format."""
    if dt is None:
        return None
    if isinstance(dt, str):
        try:
            parsed = pd.to_datetime(dt, errors="coerce")
            if pd.isna(parsed):
                return None
            return parsed.strftime("%Y-%m-%d")
        except:
            return None
    if isinstance(dt, (datetime, pd.Timestamp, pd._libs.tslibs.timestamps.Timestamp)):
        return dt.strftime("%Y-%m-%d")
    # Handle date objects
    try:
        return dt.strftime("%Y-%m-%d")
    except:
        return None

def normalize_gender(gender):
    """Map to canonical gender value."""
    if not gender:
        return "Prefer not to say"
    if isinstance(gender, str):
        if len(gender) > 30:
            return "Prefer not to say"
        mapped = GENDER_MAP.get(gender.lower()) or GENDER_MAP.get(gender)
        return mapped if mapped else "Prefer not to say"
    return "Prefer not to say"

def normalize_status(status):
    """Map to canonical status with title case."""
    if not status:
        return "Active"
    if isinstance(status, str):
        status = status.strip().title()
        status = re.sub(r'\s+', ' ', status)
        return status if status in STATUS_CANONICAL else "Active"
    return "Active"

def is_valid_dob(dob):
    """Check if DOB is valid Gregorian date with age 13-115."""
    if not dob: 
        return False
    try: 
        parsed = pd.to_datetime(dob, errors="coerce")
        if pd.isna(parsed):
            return False
        # Use naive datetime for comparison
        now = datetime.now()
        if hasattr(parsed, 'tz_localize'):
            parsed = parsed.replace(tzinfo=None)
        if parsed > now:
            return False
        age = (now - parsed).days / 365.2425
        return 13 <= age <= 115
    except: 
        return False

def is_valid_dob_at_signup(dob, created_date):
    """Check if user was 13+ at account creation."""
    if not dob or not created_date:
        return True
    try:
        dob_parsed = pd.to_datetime(dob)
        created_parsed = pd.to_datetime(created_date) if isinstance(created_date, str) else created_date
        # Remove timezone info for comparison
        if hasattr(dob_parsed, 'tz_localize'):
            dob_parsed = dob_parsed.replace(tzinfo=None)
        if hasattr(created_parsed, 'tz_localize'):
            created_parsed = created_parsed.replace(tzinfo=None)
        min_dob = created_parsed - timedelta(days=13*365.25)
        return dob_parsed <= min_dob
    except: 
        return True

def status_to_is_active(status):
    """Determine is_active boolean from status."""
    return status == "Active"

def normalize_country_code(country):
    """Normalize to ISO 3166-1 alpha-2 only."""
    iso_map = {"USA": "US", "UK": "GB", "Canada": "CA", "Australia": "AU"}
    if not country:
        return "US"
    if isinstance(country, str):
        country = country.strip().upper()
        mapped = iso_map.get(country)
        if mapped:
            return mapped
        if len(country) == 2 and country.isalpha():
            return country
    return "US"

def is_valid_postal_code(postal, country):
    """Validate postal code format for country and reject repeated digits."""
    if not postal or isinstance(postal, float):
        return True
    postal_str = str(postal).strip().upper()
    
    if len(postal_str) >= 5 and len(set(postal_str.replace(" ", "").replace("-", ""))) == 1:
        return False
    
    return True

def postal_code_for_country(country):
    """Generate valid country-specific postal code."""
    if country == "US":
        return f"{random.randint(10001, 99998)}"
    elif country == "CA":
        return f"{random.choice(string.ascii_uppercase)}{random.randint(0,9)}{random.choice(string.ascii_uppercase)} {random.randint(0,9)}{random.choice(string.ascii_uppercase)}{random.randint(0,9)}"
    elif country == "GB":
        return f"{random.choice(string.ascii_uppercase)}{random.randint(1,9)} {random.randint(0,9)}{random.choice(string.ascii_uppercase)}{random.choice(string.ascii_uppercase)}"
    elif country == "AU": 
        return f"{random.randint(1001, 9998)}"
    return f"{random.randint(10000, 99999)}"

def normalize_city(city):
    """Validate and normalize city (2-60 chars, title case, allowed chars only)."""
    if not city:
        return None
    if isinstance(city, str):
        city = city.strip().title()
        if len(city) < 2 or len(city) > 60:
            return None
        if not re.match(r"^[a-zA-Z\s\-'\.]+$", city):
            return None
        return city
    return None

def normalize_email(email):
    """Lowercase, trim, validate domain, check for dummy domains."""
    if not email:
        return None
    if isinstance(email, str):
        email = email.strip().lower()
        if not re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", email):
            return None
        if any(dummy in email for dummy in DUMMY_EMAIL_DOMAINS):
            return None
        if ".." in email or email.startswith(".") or email.endswith("."):
            return None
        if " " in email:
            return None
        return email
    return None

def normalize_phone(phone):
    """Normalize to E.164 format when possible."""
    if not phone:
        return None
    if isinstance(phone, str):
        phone = phone.strip()
        digits = re.sub(r'\D', '', phone)
        if len(digits) < 7 or len(digits) > 15:
            return None
        if digits in ["1234567890", "0000000000"] or len(set(digits)) == 1:
            return None
        if not digits.startswith("1") and len(digits) == 10:
            digits = "1" + digits
        if len(digits) == 11 and digits[0] == "1":
            return f"+{digits}"
        if len(digits) >= 7:
            return f"+{digits}" if not digits.startswith("+") else digits
    return None


def generate_messy_customer_data(num_rows=1000):
    """Generate customer data with all validation rules integrated."""
    data = []
    used_emails = set()
    used_user_ids = set()

    for i in range(num_rows):
        record = {}
        idx = i + 1

        record["user_id"] = generate_user_id(idx)
        used_user_ids.add(record["user_id"])

        gender = random.choice(["Male", "Female", "Other", "Prefer not to say"])
        record["gender"] = normalize_gender(gender)

        # FIX 1: Ensure date_of_birth is always set
        dob = fake.date_of_birth(minimum_age=18, maximum_age=80)
        dob_converted = to_iso8601_date(dob)
        record["date_of_birth"] = dob_converted

        status = random.choice(["Active", "Inactive", "Blocked", "Closed"])
        record["account_status"] = normalize_status(status)

        reg_date = fake.date_time_between(start_date="-5y", end_date="now")
        record["account_created_at"] = to_iso8601(reg_date)

        # Adjust DOB if needed to ensure user was 13+ at signup
        if record["date_of_birth"] and record["account_created_at"]: 
            if not is_valid_dob_at_signup(record["date_of_birth"], record["account_created_at"]):
                record["date_of_birth"] = to_iso8601_date(
                    pd.to_datetime(record["account_created_at"]) - timedelta(days=20*365.25)
                )

        address_id = f"ADDR_{i + 1}"
        
        if address_id and isinstance(address_id, str):
            if re.match(r"^ADDR_[1-9][0-9]*$", address_id):
                record["address_id"] = address_id
            else:
                record["address_id"] = None
        else:
            record["address_id"] = None

        city = fake.city()
        record["city"] = normalize_city(city)

        country = fake.country_code()
        record["country"] = normalize_country_code(country)

        state = random.choice([fake.state(), fake.state_abbr()])
        
        if record["country"] in ["US", "CA", "AU"] and record["address_id"]:
            if not state:
                state = fake.state_abbr()
        
        record["state_province"] = state.strip() if (state and isinstance(state, str)) else None

        postal = postal_code_for_country(record["country"])
        
        if postal and is_valid_postal_code(postal, record["country"]):
            record["postal_code"] = postal
        else:
            record["postal_code"] = None

        try:
            if reg_date:
                last_login = fake.date_time_between(start_date=reg_date, end_date="now")
            else:
                last_login = fake.date_time_between(start_date="-1y", end_date="now")
        except:
            last_login = fake.date_time_between(start_date="-1y", end_date="now")
        
        record["last_login_date"] = to_iso8601(last_login)

        record["is_active"] = status_to_is_active(record["account_status"]) if record["account_status"] else None

        # FIX 2: Ensure email is always generated and unique
        email_attempts = 0
        while email_attempts < 10: 
            email = fake.email()
            normalized_email = normalize_email(email)
            if normalized_email and normalized_email not in used_emails:
                record["email_address"] = normalized_email
                used_emails.add(normalized_email)
                break
            email_attempts += 1
        
        # If still no email after attempts, generate a guaranteed unique one
        if "email_address" not in record or record.get("email_address") is None:
            record["email_address"] = f"user{idx}_{random.randint(1000, 9999)}@mail.com"
            used_emails.add(record["email_address"])

        # FIX 3: Ensure phone is always generated
        phone_attempts = 0
        while phone_attempts < 10:
            phone = fake.phone_number()
            normalized_phone = normalize_phone(phone)
            if normalized_phone: 
                record["phone_number"] = normalized_phone
                break
            phone_attempts += 1
        
        # If still no phone, generate a guaranteed valid one
        if "phone_number" not in record or record.get("phone_number") is None:
            record["phone_number"] = f"+1{random.randint(2000000000, 9999999999)}"

        # Calculate age from date_of_birth
        if record["date_of_birth"]: 
            try:
                dob_parsed = pd.to_datetime(record["date_of_birth"])
                age = int((datetime.now() - dob_parsed).days / 365.2425)
                if 13 <= age <= 115:
                    record["age"] = age
                else:
                    record["age"] = None
            except:
                record["age"] = None
        else:
            record["age"] = None

        total_purchases = random.randint(0, 100)
        
        if isinstance(total_purchases, int) and total_purchases >= 0:
            record["total_purchases"] = total_purchases
        else:
            record["total_purchases"] = None

        ltv = round(random.uniform(0.01, 50000), 2)
        
        if record.get("total_purchases") and record["total_purchases"] > 0:
            if isinstance(ltv, (int, float)) and ltv <= 0:
                ltv = round(random.uniform(10, 50000), 2)
        
        if record.get("total_purchases") == 0:
            ltv = 0
        
        if isinstance(ltv, (int, float)) and ltv >= 0:
            record["lifetime_value"] = round(ltv, 2)
        else:
            record["lifetime_value"] = None

        # FIX 4: Ensure last_purchase_date is properly set
        if record.get("total_purchases") and record["total_purchases"] > 0:
            # Generate a date object and ensure it's converted properly
            last_purchase = fake.date_between(start_date="-1y", end_date="today")
            record["last_purchase_date"] = to_iso8601_date(last_purchase)
        else:
            # No purchases means no last purchase date
            record["last_purchase_date"] = None

        points = random.randint(0, 5000)
        
        if record.get("total_purchases") == 0:
            points = random.randint(0, 500)
        
        if isinstance(points, int) and points >= 0:
            record["loyalty_points"] = points
        else:
            record["loyalty_points"] = None

        if record["is_active"] and record["account_status"] == "Active":
            if record["last_login_date"] is None and record["last_purchase_date"] is None:
                if random.random() > 0.5:
                    record["last_login_date"] = to_iso8601(fake.date_time_between(start_date="-2y", end_date="now"))
                else:
                    last_purchase = fake.date_between(start_date="-2y", end_date="today")
                    record["last_purchase_date"] = to_iso8601_date(last_purchase)

        data.append(record)

    df = pd.DataFrame(data)
    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    """Add additional data quality issues."""
    string_cols = df.select_dtypes(include=["object"]).columns
    
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (str(x).upper() if pd.notna(x) and random.random() > 0.5 else 
                      str(x).lower() if pd.notna(x) else x)
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    for col in string_cols[:2]:
        mask = np.random.random(len(df)) < 0.01
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (str(x).replace("a", "ã").replace("e", "é") if pd.notna(x) and random.random() > 0.5 else x)
        )

    return df

df = generate_messy_customer_data(1000)
df = add_more_messiness(df)

output_file = "customers.xlsx"
df.to_excel(output_file, index=False)

print(f"Generated {len(df)} customer records")
print(f"\nColumn null counts:")
print(df.isnull().sum())
print(f"\nSample of first 5 rows:")
print(df.head())

Generated 1000 customer records

Column null counts:
user_id                0
gender                 0
date_of_birth          0
account_status         0
account_created_at     0
address_id             0
city                   0
country                0
state_province         0
postal_code            0
last_login_date        0
is_active              0
email_address          0
phone_number           0
age                    0
total_purchases        0
lifetime_value         0
last_purchase_date    11
loyalty_points         0
dtype: int64

Sample of first 5 rows:
   user_id             gender date_of_birth account_status  \
0      522             Female    1951-08-10         Active   
1      738             Female    1966-12-30         Active   
2      741               Male    1969-10-08         Closed   
3      661  Prefer not to say    1968-02-07     Inactive     
4      412               Male    1959-09-08       Inactive   

     account_created_at address_id             city country  

### Address Table Generator

In [14]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker(["en_US", "en_GB", "en_CA", "en_AU"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
ADDRESS_TYPE_CANONICAL = ["Shipping", "Billing", "Home", "Office", "Warehouse"]
COUNTRY_CODES = [
    "US",
    "GB",
    "DE",
    "FR",
    "CA",
    "AU",
    "JP",
    "CN",
    "IN",
    "BR",
    "MX",
    "IT",
    "ES",
    "NL",
]

# US states for validation
US_STATES = [
    "Alabama",
    "Alaska",
    "Arizona",
    "Arkansas",
    "California",
    "Colorado",
    "Connecticut",
    "Delaware",
    "Florida",
    "Georgia",
    "Hawaii",
    "Idaho",
    "Illinois",
    "Indiana",
    "Iowa",
    "Kansas",
    "Kentucky",
    "Louisiana",
    "Maine",
    "Maryland",
    "Massachusetts",
    "Michigan",
    "Minnesota",
    "Mississippi",
    "Missouri",
    "Montana",
    "Nebraska",
    "Nevada",
    "New Hampshire",
    "New Jersey",
    "New Mexico",
    "New York",
    "North Carolina",
    "North Dakota",
    "Ohio",
    "Oklahoma",
    "Oregon",
    "Pennsylvania",
    "Rhode Island",
    "South Carolina",
    "South Dakota",
    "Tennessee",
    "Texas",
    "Utah",
    "Vermont",
    "Virginia",
    "Washington",
    "West Virginia",
    "Wisconsin",
    "Wyoming",
]

US_STATE_ABBR = [
    "AL",
    "AK",
    "AZ",
    "AR",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "ID",
    "IL",
    "IN",
    "IA",
    "KS",
    "KY",
    "LA",
    "ME",
    "MD",
    "MA",
    "MI",
    "MN",
    "MS",
    "MO",
    "MT",
    "NE",
    "NV",
    "NH",
    "NJ",
    "NM",
    "NY",
    "NC",
    "ND",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VT",
    "VA",
    "WA",
    "WV",
    "WI",
    "WY",
]

# Generate customer and supplier IDs
customer_ids = [
    f"CUST_{i + 1}" for i in range(1000)
]
supplier_ids = [
    f"SUP_{i + 1}" for i in range(100)
]


def generate_messy_address_data(num_rows=2000):
    data = []
    used_address_ids = []
    default_addresses = {}  # Track defaults per (owner_id, address_type)

    for i in range(num_rows):
        record = {}

        # address_id: Primary key, format ^ADDR_[1-9][0-9]*$, unique
        addr_id = f"ADDR_{i + 1}"

        used_address_ids.append(addr_id)
        record["address_id"] = addr_id

        # Decide owner type (80% customer, 20% supplier)
        is_supplier_owned = random.random() < 0.20

        # user_id/owner_id: Mandatory, FK to customers or suppliers
        if is_supplier_owned:
            owner_id = random.choice(supplier_ids)
        else:
            owner_id = random.choice(customer_ids)
        record["owner_id"] = owner_id

        # address_line1: Mandatory, 5-100 chars, must contain street number + name
        street1 = fake.street_address()
        record["address_line1"] = street1

        # address_line2: Always populated
        street2 = fake.secondary_address()
        record["address_line2"] = street2

        # city: Mandatory, 2-60 chars, no digits-only
        city = fake.city()
        record["city"] = city

        # state_province: Mandatory for US/CA/AU, valid subdivisions
        state = fake.state()
        record["state_province"] = state

        # postal_code: Country-specific format, mandatory
        postal = fake.postcode()
        record["postal_code"] = postal

        # country: Mandatory, ISO 3166-1 alpha-2 or official names
        country = fake.country()
        record["country"] = country

        # address_type: Canonical set
        if is_supplier_owned:
            addr_type = random.choice(["Warehouse", "Office"])
        else:
            addr_type = random.choice(["Shipping", "Billing", "Home", "Office"])
        record["address_type"] = addr_type

        # is_default: Boolean, uniqueness per (owner_id, address_type)
        key = (owner_id, addr_type)
        if key in default_addresses:
            is_default = False  # Already have a default
        else:
            is_default = random.choices([True, False], weights=[30, 70], k=1)[0]
            if is_default:
                default_addresses[key] = True
        record["is_default"] = is_default

        # phone_number: Always populated
        phone = fake.phone_number()
        record["phone_number"] = phone

        # coordinates: Always populated
        lat = float(fake.latitude())
        lon = float(fake.longitude())
        record["latitude"] = lat
        record["longitude"] = lon

        # resident_name: Always populated
        if is_supplier_owned:
            name = random.choice(
                ["Receiving Department", "Warehouse Manager", "Shipping Dept"]
            )
        else:
            name = fake.name()
        record["resident_name"] = name

        # created_at: Always populated
        created = fake.date_time_between(start_date="-2y", end_date="now")
        record["created_at"] = created

        # updated_at: Always populated
        updated = fake.date_time_between(
            start_date=record["created_at"], end_date="now"
        )
        record["updated_at"] = updated

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_address_data(2000)
df = add_more_messiness(df)

output_file = "addresses.xlsx"
df.to_excel(output_file, index=False)

### Products Table Generator

In [15]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
AVAILABILITY_STATUS_CANONICAL = [
    "In Stock",
    "Discontinued",
    "Out of Stock",
    "Pre-order",
    "Archive",
]
COLORS_CANONICAL = [
    "Black",
    "White",
    "Blue",
    "Red",
    "Gray",
    "Silver",
    "Gold",
    "Green",
    "Navy",
    "Pink",
    "Purple",
    "Orange",
    "Yellow",
    "Brown",
    "Beige",
    "Space Gray",
]
CLOTHING_SIZES = ["XS", "S", "M", "L", "XL", "XXL", "XXXL"]
SHOE_SIZES_US = [
    "6",
    "6.5",
    "7",
    "7.5",
    "8",
    "8.5",
    "9",
    "9.5",
    "10",
    "10.5",
    "11",
    "11.5",
    "12",
    "13",
]
ELECTRONIC_SIZES = [
    '13"',
    '14"',
    '15.6"',
    '17"',
    '21"',
    '24"',
    '27"',
    '32"',
    "128GB",
    "256GB",
    "512GB",
    "1TB",
]
MATERIALS_FOOTWEAR = [
    "Leather",
    "Synthetic",
    "Canvas",
    "Mesh",
    "Rubber",
    "Suede",
    "Nylon",
]
MATERIALS_CLOTHING = [
    "Cotton",
    "Polyester",
    "Wool",
    "Silk",
    "Linen",
    "Denim",
    "Nylon",
    "Spandex",
]
MATERIALS_ELECTRONICS = [
    "Aluminum",
    "Plastic",
    "Glass",
    "Carbon Fiber",
    "Steel",
    "Magnesium Alloy",
]

BRANDS = [
    "Nike",
    "Adidas",
    "Apple",
    "Samsung",
    "Sony",
    "Dell",
    "HP",
    "Lenovo",
    "Microsoft",
    "Google",
    "Amazon",
    "LG",
    "Panasonic",
    "Canon",
    "Nikon",
    "Bose",
    "JBL",
    "Reebok",
    "Puma",
    "Under Armour",
]

PRODUCT_TYPES = [
    "Laptop",
    "Smartphone",
    "Tablet",
    "Headphones",
    "Smartwatch",
    "Camera",
    "Running Shoes",
    "Basketball Shoes",
    "Training Shoes",
    "T-Shirt",
    "Jacket",
    "Backpack",
    "Monitor",
    "Keyboard",
]

CATEGORY_NAMES = [
    "Electronics",
    "Footwear",
    "Apparel",
    "Accessories",
    "Sports",
    "Home & Office",
    "Audio",
    "Gaming",
    "Photography",
    "Fitness",
    "Outdoor",
    "Travel",
    "Technology",
    "Fashion",
]

SUB_CATEGORIES = {
    "Electronics": ["Computers", "Mobile", "Tablets", "Wearables", "Accessories"],
    "Footwear": ["Running", "Basketball", "Casual", "Formal", "Outdoor"],
    "Apparel": ["Shirts", "Pants", "Jackets", "Sportswear", "Formal"],
    "Accessories": ["Bags", "Belts", "Watches", "Jewelry", "Hats"],
    "Sports": ["Equipment", "Clothing", "Shoes", "Accessories", "Nutrition"],
    "Audio": ["Headphones", "Speakers", "Earbuds", "Microphones", "Amplifiers"],
    "Gaming": ["Consoles", "Controllers", "Headsets", "Keyboards", "Mice"],
}


def generate_messy_product_data(num_rows=1000):
    data = []
    used_product_ids = []
    used_skus = []

    # Generate category and supplier pools
    categories = [
        f"CAT_{i + starting_category_index}" for i in range(number_of_categories)
    ]
    suppliers = [
        f"SUPP_{i + starting_supplier_index}" for i in range(number_of_suppliers)
    ]

    for i in range(num_rows):
        record = {}

        # prod_id: Primary key, positive integer, unique
        if i % 53 == 0 and used_product_ids:
            prod_id = random.choice(used_product_ids)  # Duplicate violation
        else:
            prod_id = starting_product_index + i
            used_product_ids.append(prod_id)
        record["prod_id"] = prod_id

        # Determine product category for related attributes
        product_category = random.choice(
            ["electronics", "footwear", "clothing", "general"]
        )
        is_digital = (
            product_category == "electronics" and random.random() < 0.1
        )  # 10% of electronics are digital

        # product_name: Mandatory, 5-150 chars, descriptive
        if i % 41 == 0:
            # Very long name violation
            brand = random.choice(BRANDS)
            product = random.choice(PRODUCT_TYPES)
            name = f"{brand} {product} Premium Edition with Extra Features and Extended Warranty Limited Time Offer Special Bundle Pack"
        elif i % 51 == 0:
            # Short name violation (< 5 chars)
            name = random.choice(["ABC", "XYZ", "Pro", "Air"])
        elif i % 61 == 0:
            # Name with special characters
            brand = random.choice(BRANDS)
            product = random.choice(PRODUCT_TYPES)
            name = f"{brand} {product}™ #{i}"
        else:
            brand = random.choice(BRANDS)
            product = random.choice(PRODUCT_TYPES)
            model = (
                random.choice(["Pro", "Air", "Ultra", "Max", "Plus", "Elite"])
                if random.random() > 0.3
                else ""
            )
            name = f"{brand} {product} {model}".strip()
        record["product_name"] = name

        # stock_code (SKU): Mandatory, unique, canonical pattern ^[A-Z0-9]{2,5}-[A-Z0-9]{2,10}-[0-9]{3,6}$
        if i % 37 == 0 and used_skus:
            sku = random.choice(used_skus)  # Duplicate violation
        elif i % 57 == 0:
            sku = f"SKU-{i}!@#"  # Special chars violation
        elif i % 67 == 0:
            # Wrong format - lowercase
            brand_code = name[:2].upper()
            sku = f"{brand_code.lower()}-el{str(i).zfill(4)}-{random.choice(COLORS_CANONICAL)[:3].lower()}"
        else:
            # Valid SKU format: ^[A-Z0-9]{2,5}-[A-Z0-9]{2,10}-[0-9]{3,6}$
            brand_code = name[:2].upper()
            category_code = random.choice(["EL", "CL", "SP", "AC", "HM"])
            sku = f"{brand_code}-{category_code}{str(i).zfill(4)}-{str(random.randint(100, 999999)).zfill(6)}"
            used_skus.append(sku)
        record["stock_code"] = sku

        # category_ref: Mandatory, format ^CAT_[1-9][0-9]*$, FK
        if i % 39 == 0:
            category_id = f"CAT_{9999}"  # FK violation
        else:
            category_id = random.choice(categories)
        record["category_ref"] = category_id

        # cat_name: Must match category_ref
        if i % 56 == 0:
            cat_name = random.choice(["Electronix", "Footware", "Aparrel"])  # Typos
        else:
            # Valid: match with product category
            if product_category == "footwear":
                cat_name = "Footwear"
            elif product_category == "clothing":
                cat_name = "Apparel"
            elif product_category == "electronics":
                cat_name = "Electronics"
            else:
                cat_name = random.choice(CATEGORY_NAMES)
        record["cat_name"] = cat_name

        # sub_cat: Must be valid for cat_name
        if i % 43 == 0:
            # Inconsistent with main category
            sub_cat = random.choice(["Computers", "Running", "Shirts"])
        elif i % 63 == 0:
            sub_cat = random.choice(["Runing", "Casul", "Moble"])  # Typos
        else:
            # Valid: appropriate subcategory based on category
            if cat_name in SUB_CATEGORIES:
                sub_cat = random.choice(SUB_CATEGORIES[cat_name])
            elif product_category == "footwear":
                sub_cat = random.choice(["Running", "Basketball", "Casual"])
            elif product_category == "clothing":
                sub_cat = random.choice(["Shirts", "Pants", "Jackets"])
            elif product_category == "electronics":
                sub_cat = random.choice(["Computers", "Mobile", "Tablets"])
            else:
                sub_cat = random.choice(["General", "Miscellaneous"])
        record["sub_cat"] = sub_cat

        # brand: Mandatory, standardized
        if i % 34 == 0:
            brand_val = random.choice(
                ["nike", "NIKE", "Nike Inc.", "Nike®"]
            )  # Inconsistent
        elif i % 44 == 0:
            brand_val = random.choice(["Addidas", "Appl", "Samung"])  # Typos
        else:
            brand_val = random.choice(BRANDS)
        record["brand"] = brand_val

        # supp_id: Mandatory, format ^SUPP_[1-9][0-9]*$, FK
        if i % 32 == 0:
            supplier = f"SUPP_{999}"  # FK violation
        else:
            supplier = random.choice(suppliers)
        record["supp_id"] = supplier

        # unit_cost: Decimal >= 0, typically <= retail_price
        # if i % 45 == 0:
        #     cost = round(random.uniform(-100, -1), 2)  # Negative violation
        # elif i % 55 == 0:
        #     cost = random.choice([0, 999999.99, 0.001, -9999])  # Extreme
        # else:
            # Valid cost based on product category
        if product_category == "electronics":
            cost = round(random.uniform(50, 800), 2)
        elif product_category == "footwear":
            cost = round(random.uniform(20, 150), 2)
        elif product_category == "clothing":
            cost = round(random.uniform(10, 80), 2)
        else:
            cost = round(random.uniform(10, 300), 2)
        record["unit_cost"] = cost
        product_cost_prices[prod_id] = cost

        # retail_price: Decimal >= 0, typically >= unit_cost, markup 20-300%
        # if i % 48 == 0:
        #     # Price < cost violation
        #     if isinstance(cost, (int, float)) and cost > 0:
        #         price = round(cost * 0.5, 2)
        #     else:
        #         price = round(random.uniform(1, 10), 2)
        # elif i % 58 == 0:
        #     price = random.choice([0, 999999.99, 0.01, -100])  # Extreme/negative
        # elif i % 68 == 0:
        #     # Unrealistic markup (> 300%)
        #     if isinstance(cost, (int, float)) and cost > 0:
        #         price = round(cost * 5, 2)
        #     else:
        #         price = round(random.uniform(5000, 10000), 2)
        # else:
            # Valid: retail_price >= unit_cost, markup 20-300%
        if isinstance(cost, (int, float)) and cost > 0:
            markup = random.uniform(1.2, 3.0)
            price = round(cost * markup, 2)
        else:
            price = round(random.uniform(20, 1000), 2)
        record["retail_price"] = price
        product_prices[prod_id] = price

        # release_date: YYYY-MM-DD, <= today unless Pre-order
        if i % 40 == 0:
            # String format variations
            launch_date = fake.date_between(start_date="-5y", end_date="today")
            formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            launch = launch_date.strftime(random.choice(formats))
        elif i % 60 == 0:
            launch = fake.date_between(
                start_date="today", end_date="+1y"
            )  # Future (for pre-order)
        elif i % 70 == 0:
            launch = fake.date_between(
                start_date="-50y", end_date="-30y"
            )  # Very old violation
        else:
            launch = fake.date_between(start_date="-5y", end_date="today")
        record["release_date"] = launch

        # digital_product: Boolean
        if i % 52 == 0:
            digital_val = random.choice(
                ["Y", "N", "Yes", "No", "1", "0"]
            )  # String boolean
        else:
            digital_val = is_digital
        record["digital_product"] = digital_val

        # weight: Decimal > 0 for physical, 0/NULL for digital
        if i % 43 == 0:
            weight = round(random.uniform(-10, -0.001), 3)  # Negative violation
        elif i % 53 == 0:
            weight = random.choice([0, 99999.999, 0.0001])  # Extreme
        elif i % 73 == 0:
            # Wrong weight for category
            if product_category == "electronics":
                weight = round(random.uniform(50, 100), 3)  # Too heavy for electronics
            else:
                weight = round(random.uniform(0.001, 0.01), 3)  # Too light
        elif is_digital:
            weight = 0  # Digital products have 0 weight
        else:
            # Valid weight based on category
            if product_category == "electronics":
                weight = round(random.uniform(0.1, 5), 3)
            elif product_category == "footwear":
                weight = round(random.uniform(0.2, 1.5), 3)
            elif product_category == "clothing":
                weight = round(random.uniform(0.1, 2), 3)
            else:
                weight = round(random.uniform(0.1, 10), 3)
        record["weight"] = weight

        # dimensions: LxWxH format, NULL for digital
        if i % 47 == 0:
            # Wrong separators
            l, w, h = (
                random.randint(5, 100),
                random.randint(5, 100),
                random.randint(5, 100),
            )
            dimensions = random.choice([f"{l}-{w}-{h}", f"{l}/{w}/{h}", f"{l},{w},{h}"])
        elif i % 57 == 0:
            dimensions = random.choice(["0x0x0", "-10x-10x-10"])  # Invalid values
        elif is_digital:
            dimensions = "0x0x0"  # Digital products have no dimensions
        else:
            # Valid dimensions based on category
            if product_category == "electronics":
                l, w, h = (
                    random.randint(10, 40),
                    random.randint(10, 30),
                    random.randint(1, 10),
                )
            elif product_category == "footwear":
                l, w, h = (
                    random.randint(25, 35),
                    random.randint(15, 20),
                    random.randint(10, 15),
                )
            else:
                l, w, h = (
                    random.randint(5, 50),
                    random.randint(5, 50),
                    random.randint(5, 50),
                )
            dimensions = f"{l}x{w}x{h}"
        record["dimensions"] = dimensions

        # color: Standardized lookup values
        if i % 41 == 0:
            color_choice = random.choice(COLORS_CANONICAL)
            color = random.choice(
                [color_choice.upper(), color_choice.lower()]
            )  # Case issues
        elif i % 51 == 0:
            color = f"{random.choice(COLORS_CANONICAL)}/{random.choice(COLORS_CANONICAL)}"  # Multiple
        elif i % 71 == 0:
            color = random.choice(["Balck", "Whtie", "Grey", "Blu"])  # Typos
        else:
            if product_category == "electronics":
                color = random.choice(
                    ["Black", "Silver", "White", "Space Gray", "Gold"]
                )
            else:
                color = random.choice(COLORS_CANONICAL)
        record["color"] = color

        # size: Category-appropriate format
        if i % 46 == 0:
            # Wrong format for category
            if product_category == "footwear":
                size = random.choice(CLOTHING_SIZES)  # Wrong
            elif product_category == "clothing":
                size = random.choice(SHOE_SIZES_US)  # Wrong
            else:
                size = str(random.randint(1, 100))
        elif i % 66 == 0:
            size = random.choice(["Smal", "Mediun", "Larg"])  # Typos
        else:
            # Valid size for category
            if product_category == "footwear":
                size = random.choice(SHOE_SIZES_US)
            elif product_category == "clothing":
                size = random.choice(CLOTHING_SIZES)
            elif product_category == "electronics":
                size = random.choice(ELECTRONIC_SIZES)
            else:
                size = random.choice(["Small", "Medium", "Large"])
        record["size"] = size

        # material: Category-appropriate, standardized
        if i % 44 == 0:
            material = random.choice(["Lether", "Cotten", "Pollyester"])  # Typos
        elif i % 64 == 0:
            # Wrong material for category
            if product_category == "footwear":
                material = random.choice(MATERIALS_ELECTRONICS)
            elif product_category == "electronics":
                material = random.choice(MATERIALS_CLOTHING)
            else:
                material = random.choice(MATERIALS_FOOTWEAR)
        else:
            # Valid material for category
            if product_category == "footwear":
                material = random.choice(MATERIALS_FOOTWEAR)
            elif product_category == "clothing":
                material = random.choice(MATERIALS_CLOTHING)
            elif product_category == "electronics":
                material = random.choice(MATERIALS_ELECTRONICS)
            else:
                material = random.choice(MATERIALS_FOOTWEAR + MATERIALS_CLOTHING)
        record["material"] = material

        # availability_status: Canonical set, consistent with inventory/release_date
        if i % 38 == 0:
            status = random.choice(["active", "ACTIVE", "1", "A"])  # Case issues
        elif i % 48 == 0:
            status = random.choice(["Available", "Unavailable", "Sold Out"])  # Invalid
        elif i % 58 == 0:
            status = random.choice(["Activ", "Discontined", "Out of Stok"])  # Typos
        else:
            status = random.choices(
                AVAILABILITY_STATUS_CANONICAL, weights=[50, 10, 15, 10, 15], k=1
            )[0]
        record["availability_status"] = status

        # inventory_qty: Integer >= 0, consistent with availability_status
        if i % 50 == 0:
            stock = random.randint(-100, -1)  # Negative violation
        elif i % 60 == 0:
            stock = random.choice([999999, 0.5, -9999])  # Extreme
        elif i % 75 == 0:
            # Inconsistent: Out of Stock but has inventory
            if status == "Out of Stock":
                stock = random.randint(50, 200)  # Violation
            else:
                stock = 0
        else:
            # Valid: consistent with availability_status
            if status == "Out of Stock":
                stock = 0
            elif status == "Pre-order":
                stock = 0  # Pre-order typically has 0 inventory
            elif status == "Active":
                stock = random.randint(10, 500)
            else:
                stock = random.randint(0, 100)
        record["inventory_qty"] = stock

        # avg_customer_rating: 0.0-5.0, consistent with review_count
        if i % 42 == 0:
            rating = random.choice([-1, 6, 10, 999])  # Out of range
        elif i % 52 == 0:
            rating = round(random.uniform(0, 5), 5)  # Too many decimals
        else:
            rating = round(random.uniform(1, 5), 2)
        record["avg_customer_rating"] = rating

        # review_count: Integer >= 0, consistent with rating
        if i % 45 == 0:
            reviews = random.randint(-100, -1)  # Negative violation
        elif i % 65 == 0:
            # Inconsistent: rating exists but 0 reviews
            if isinstance(rating, (int, float)) and 1 <= rating <= 5:
                reviews = 0  # Violation
            else:
                reviews = random.randint(1, 100)
        else:
            # Valid: consistent with rating
            if isinstance(rating, (int, float)) and rating >= 4:
                reviews = random.randint(10, 1000)
            else:
                reviews = random.randint(1, 100)
        record["review_count"] = reviews

        # discount_pct: 0-100, financial consistency
        if i % 53 == 0:
            discount = random.choice([-50, 150, 999])  # Out of range
        elif i % 69 == 0:
            # Discount makes price negative violation
            discount = 110
        elif i % 78 == 0:
            # Discount on pre-order violation
            if status == "Pre-order":
                discount = random.randint(30, 50)  # Unusual for pre-order
            else:
                discount = 0
        else:
            # Valid: 0-100, typically 0-50 for active products
            if status == "Pre-order":
                discount = 0  # Pre-order typically no discount
            elif status == "Discontinued":
                discount = random.randint(20, 70)  # Clearance
            else:
                discount = random.choices(
                    [0, 5, 10, 15, 20, 25, 30], weights=[40, 15, 15, 10, 10, 5, 5], k=1
                )[0]
        record["discount_pct"] = discount

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_product_data(number_of_products)
df = add_more_messiness(df)

output_file = "products.xlsx"
df.to_excel(output_file, index=False)

### Categories Table Generator

In [16]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, timezone
import random
import string
import re

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# System launch date for lower bound validation
SYSTEM_LAUNCH_DATE = datetime(2010, 1, 1, tzinfo=timezone.utc)
RESERVED_WORDS = {"admin", "cart", "checkout", "api", "search", "login", "account", "profile"}

# Controlled vocabulary for sub_cat
VALID_SUBCATEGORIES = {
    "Premium", "Standard", "Budget", "Professional", "Consumer", "Industrial",
    "Retail", "Wholesale", "Limited Edition", "Regular", "Special Offer",
    "Clearance", "New Arrival", "Best Seller", "Featured", "Sale", "On Discount"
}

def generate_cat_id(index):
    """Generate unique integer cat_id (> 0)."""
    return index

def to_iso8601(dt):
    """Convert to ISO-8601 format with UTC timezone."""
    if dt is None:
        return None
    if isinstance(dt, str):
        try:
            parsed = pd.to_datetime(dt, errors="coerce")
            if pd.isna(parsed):
                return None
            return parsed.strftime("%Y-%m-%dT%H:%M:%SZ")
        except:
            return None
    if isinstance(dt, (datetime, pd.Timestamp)):
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return None

def is_valid_cat_name(name):
    """Validate category name: 3-60 chars, no placeholders, meaningful."""
    if not name or (isinstance(name, str) and name.lower().strip() in ["n/a", "null", "test", "category", "misc", "uncategorized", "other", "general", ""]):
        return False
    if isinstance(name, str):
        name = name.strip()
        if len(name) < 3 or len(name) > 60:
            return False
        # Check for HTML tags or control characters
        if "<" in name or ">" in name or chr(0) in name:
            return False
        # Reject repeated punctuation
        if "!!!!" in name or "???" in name or "---" in name:
            return False
        return True
    return False

def normalize_cat_name(name):
    """Normalize category name: title case, trim spaces."""
    if not name:
        return None
    if isinstance(name, str):
        name = name.strip()
        if not is_valid_cat_name(name):
            return None
        # Title case (preserve brand casing like iPhone would need manual override)
        return name.title()
    return None

def normalize_subcategory(subcat):
    """Normalize sub_cat to controlled vocabulary."""
    if not subcat or (isinstance(subcat, str) and subcat.lower().strip() in ["n/a", "null", "none", "general", "", "all", "unknown"]):
        return None
    if isinstance(subcat, str):
        subcat = subcat.strip().title()
        # Map to valid vocabulary
        for valid in VALID_SUBCATEGORIES:
            if valid.lower() == subcat.lower():
                return valid
        # If not in vocabulary, reject
        return None
    return None

def is_valid_url_slug(slug):
    """Validate URL slug: lowercase a-z0-9 with single hyphens, 2-80 chars, no reserved words."""
    if not slug:
        return False
    if isinstance(slug, str):
        slug = slug.strip().lower()
        if len(slug) < 2 or len(slug) > 80:
            return False
        # Check format: ^[a-z0-9]+(?:-[a-z0-9]+)*$
        if not re.match(r"^[a-z0-9]+(?:-[a-z0-9]+)*$", slug):
            return False
        # Check for reserved words
        if slug in RESERVED_WORDS or slug.split('-')[0] in RESERVED_WORDS:
            return False
        # Check for file-like endings
        if slug.endswith(('.php', '.html', '.asp', '.jsp')):
            return False
        return True
    return False

def generate_url_slug(cat_name):
    """Generate URL slug from category name."""
    if not cat_name:
        return None
    slug = cat_name.lower().replace(" ", "-").replace("&", "and")
    slug = re.sub(r'[^a-z0-9-]', '', slug)
    slug = re.sub(r'-+', '-', slug)
    slug = slug.strip('-')
    if is_valid_url_slug(slug):
        return slug
    return None

def normalize_bool(value):
    """Normalize to boolean: Y/N, yes/no, true/false, 1/0, T/F."""
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        value_lower = value.lower().strip()
        if value_lower in ['y', 'yes', 'true', 't', '1']:
            return True
        elif value_lower in ['n', 'no', 'false', 'f', '0']:
            return False
    if isinstance(value, (int, float)):
        return bool(value)
    return None

def is_valid_hierarchy_level(level):
    """Validate hierarchy_level: 0-3 typically."""
    if level is None:
        return True
    try:
        level_int = int(level)
        return 0 <= level_int <= 10  # Allow deeper trees but flag extreme values
    except:
        return False

def breadcrumb_from_hierarchy(cat_name, parent_name=None, level=None):
    """Generate breadcrumb path from hierarchy info."""
    if not cat_name:
        return None
    if level == 0 or parent_name is None:
        return cat_name
    if parent_name:
        return f"{parent_name} > {cat_name}"
    return cat_name


def generate_messy_category_data(num_rows=200):
    """Generate category data with all 12 validation rules integrated."""
    data = []
    used_cat_ids = set()
    used_url_slugs = set()
    used_names_by_parent = {}  # Track names per parent for uniqueness rule
    category_hierarchy = {}  # Track hierarchy for validation

    for i in range(num_rows):
        record = {}
        idx = starting_category_index + i

        # 1.cat_id: Pure integer, must be > 0, unique (NO DUPLICATES)
        cat_id = generate_cat_id(idx)
        record["cat_id"] = cat_id
        used_cat_ids.add(cat_id)

        # 9.hierarchy_level: Integer >= 0, determines parent/child rules
        # Generate level: ~60% root, ~30% level 1, ~10% level 2+
        rand = random.random()
        if rand < 0.6:
            level = 0
        elif rand < 0.9:
            level = 1
        else:
            level = random.choice([2, 3])
        
        record["hierarchy_level"] = level

        # 12.parent_cat_id: Must be NULL iff level = 0, referential integrity
        parent_cat_id = None
        if level and level > 0:
            # Child nodes must have parent
            if len([c for c in category_hierarchy.values() if c['level'] == level - 1]) > 0:
                possible_parents = [c['id'] for c in category_hierarchy.values() if c['level'] == level - 1]
                parent_cat_id = random.choice(possible_parents)
            elif used_cat_ids and len(used_cat_ids) > 1:
                parent_cat_id = random.choice(list(used_cat_ids)[:-1])
        elif level == 0:
            # Root nodes must NOT have parent
            parent_cat_id = None
        else:
            # No level info - randomly decide
            if random.random() > 0.7 and used_cat_ids and len(used_cat_ids) > 1:
                parent_cat_id = random.choice(list(used_cat_ids)[:-1])

        # Validation: no self-reference
        if parent_cat_id == cat_id:
            parent_cat_id = None

        record["parent_cat_id"] = parent_cat_id

        # 2.cat_name: NOT NULL, 3-60 chars, unique within parent, no placeholders
        if i % 35 == 0:
            name = "Test" * 20  # Exceeds 60 chars
        elif i % 50 == 0:
            name = random.choice(["electronics", "ELECTRONICS", "ElEcTrOnIcS"])
        else:
            name = random.choice([
                "Electronics", "Clothing", "Sports & Outdoors", "Home & Garden",
                "Books & Media", "Computers", "Mobile Devices", "Audio",
                "Cameras", "Gaming", "Accessories", "Men's Clothing",
                "Women's Clothing", "Kids' Clothing", "Shoes", "Fitness",
                "Outdoor Recreation", "Team Sports", "Water Sports",
                "Winter Sports", "Furniture", "Kitchen", "Bedroom",
                "Bathroom", "Garden", "Fiction", "Non-Fiction"
            ])

        # Normalize name
        normalized_name = normalize_cat_name(name)
        
        # Enforce NOT NULL
        if normalized_name is None:
            name = random.choice(["Electronics", "Clothing", "Sports"])
        else:
            name = normalized_name

        # Check uniqueness within parent
        parent_key = str(parent_cat_id) if parent_cat_id else "root"
        if parent_key not in used_names_by_parent:
            used_names_by_parent[parent_key] = set()

        # If already used, regenerate
        if name in used_names_by_parent[parent_key]:
            if i % 3 == 0: # Keep some duplicates for testing
                pass
            else:
                name = f"{name} {random.randint(1, 999)}"

        used_names_by_parent[parent_key].add(name)
        record["cat_name"] = name

        # 3.sub_cat: From controlled vocabulary
        subcat = random.choice(list(VALID_SUBCATEGORIES))
        record["sub_cat"] = subcat

        # 5.url_slug: Required for public, unique, format validation, reserved words check
        if i % 55 == 0:
            slug = "admin-section"  # Reserved word prefix
        elif i % 65 == 0:
            slug = "category-page"  # File-like pattern
        else:
            slug = generate_url_slug(name) if name else f"category-{i}"

        # Validate format
        if slug and not is_valid_url_slug(slug):
            slug = f"category-{i}"

        # Enforce uniqueness
        if slug and slug in used_url_slugs:
            if random.random() > 0.8:
                pass  # Keep some duplicates for testing
            else:
                slug = f"{slug}-{random.randint(1, 999)}"

        used_url_slugs.add(slug)
        record["url_slug"] = slug

        # 6.active_flag: Boolean only, hierarchy dependency
        if i % 35 == 0:
            active = random.choice(["Y", "N", "Yes", "No", "1", "0"])
            active = normalize_bool(active)
        else:
            active = random.choice([True, True, True, True, False])  # 80% active

        # Hierarchy dependency: If parent is inactive, child should be inactive
        if parent_cat_id and parent_cat_id in category_hierarchy:
            if not category_hierarchy[parent_cat_id]['active']:
                if random.random() > 0.3: # 70% enforce, 30% allow violation
                    active = False

        record["active_flag"] = active

        # 8.category_desc: 20-500 chars
        desc = fake.sentence(nb_words=random.randint(4, 20))

        # Validate: should be 20-500 chars
        if isinstance(desc, str):
            desc = desc.strip()
            if len(desc) < 20:
                # Pad with more text
                desc = desc + " " + fake.sentence(nb_words=5)
            if len(desc) > 500:
                desc = desc[:500]

        record["category_desc"] = desc

        # 4.display_sequence: Integer >= 0, sibling uniqueness
        sequence = random.randint(0, 1000)
        record["display_sequence"] = sequence

        # 10.product_count: Integer >= 0, active realism
        count = random.randint(0, 500)

        # Active realism: If active, product_count should usually be > 0
        if active and count == 0:
            if random.random() > 0.7: # 70% enforce
                count = random.randint(1, 100)

        record["product_count"] = count

        # 11.breadcrumb_path: Required for non-root, correct structure
        if level == 0:
            breadcrumb = name  # Root just has its name
        else:
            parent_name = None
            if parent_cat_id in category_hierarchy:
                parent_name = category_hierarchy[parent_cat_id].get('name')
            
            if parent_name:
                breadcrumb = f"{parent_name} > {name}"
            else:
                breadcrumb = name

        record["breadcrumb_path"] = breadcrumb

        # 7.date_created: ISO-8601, <= now(), >= system launch date
        if i % 40 == 0:
            created = fake.date_time_between(start_date="+1y", end_date="+2y")  # Future date
        elif i % 50 == 0:
            created = fake.date_time_between(start_date="-100y", end_date="-50y")  # Too old
        else:
            created = fake.date_time_between(start_date=SYSTEM_LAUNCH_DATE, end_date="now")

        # Convert to ISO-8601 and validate
        created_iso = to_iso8601(created)
        if created_iso:
            created_dt = pd.to_datetime(created_iso)
            if created_dt > datetime.now(timezone.utc):
                created_iso = to_iso8601(datetime.now(timezone.utc))
            if created_dt < SYSTEM_LAUNCH_DATE:
                created_iso = to_iso8601(SYSTEM_LAUNCH_DATE)

        record["date_created"] = created_iso

        # Store category info for hierarchy validation
        category_hierarchy[cat_id] = {
            'id': cat_id,
            'name': name,
            'level': level,
            'active': active,
            'parent': parent_cat_id
        }

        data.append(record)

    df = pd.DataFrame(data)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    """Add additional data quality issues."""
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (str(x).upper() if pd.notna(x) and random.random() > 0.5 else 
                      str(x).lower() if pd.notna(x) else x)
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


if __name__ == "__main__":
    df = generate_messy_category_data(number_of_categories)
    df = add_more_messiness(df)

    output_file = "categories.xlsx"
    df.to_excel(output_file, index=False)

### Wishlist Table Generator

In [17]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
PRIORITY_CANONICAL = ["Low", "Medium", "High"]
LIST_NAME_CANONICAL = ["Wishlist", "Gift Ideas", "Future Purchases", "Dream Items"]
ADD_SOURCE_CANONICAL = [
    "Web",
    "Mobile App",
    "Android",
    "iOS",
    "Social Media",
    "QR Code",
    "Email Link",
]

# Generate product prices for consistency
# product_prices = {
#     f"PROD_{i + starting_product_index}": round(random.uniform(10, 500), 2)
#     for i in range(number_of_products)
# }


def generate_messy_wishlist_data(
    num_rows=2000, customer_id_format="CUST", product_id_format="PROD"
):
    data = []
    used_wishlist_ids = []

    customer_ids = [
        f"CUST_{i + starting_customer_index}" for i in range(number_of_customers)
    ]
    product_ids = [
        f"PROD_{i + starting_product_index}" for i in range(number_of_products)
    ]

    customer_product_pairs = {}

    for i in range(num_rows):
        record = {}

        # wish_id: Primary key, positive integer, unique
        if i % 53 == 0 and used_wishlist_ids:
            wish_id = random.choice(used_wishlist_ids)  # Duplicate violation
        else:
            wish_id = starting_wishlist_index + i
            used_wishlist_ids.append(wish_id)
        record["wish_id"] = wish_id

        # user_id: Mandatory, FK to customers
        if i % 61 == 0:
            cust_id = f"CUST_{99999}"  # FK violation
        else:
            cust_id = random.choice(customer_ids)
        record["user_id"] = cust_id

        # item_id: Mandatory, FK to products
        if i % 57 == 0:
            prod_id = f"PROD_{9999}"  # FK violation
        else:
            # Check for duplicate customer-product pairs
            if i % 31 == 0 and cust_id in customer_product_pairs:
                if customer_product_pairs[cust_id]:
                    prod_id = random.choice(customer_product_pairs[cust_id])
                else:
                    prod_id = random.choice(product_ids)
            else:
                prod_id = random.choice(product_ids)

        if cust_id and prod_id:
            if cust_id not in customer_product_pairs:
                customer_product_pairs[cust_id] = []
            customer_product_pairs[cust_id].append(prod_id)
        record["item_id"] = prod_id

        # Get product price for this item
        base_price = product_prices.get(prod_id, round(random.uniform(10, 500), 2))

        # date_added: <= now(), earliest event timestamp
        if i % 37 == 0:
            added_date = fake.date_time_between(start_date="-2y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y"]
            added = added_date.strftime(random.choice(formats))
        elif i % 47 == 0:
            added = int(
                fake.date_time_between(start_date="-2y", end_date="now").timestamp()
            )
        elif i % 57 == 0:
            added = fake.date_time_between(
                start_date="+1m", end_date="+1y"
            )  # Future violation
        elif i % 67 == 0:
            added = fake.date_time_between(
                start_date="-10y", end_date="-5y"
            )  # Very old
        else:
            added = fake.date_time_between(start_date="-2y", end_date="now")
        record["date_added"] = added

        # Determine outcome: purchased, removed, or active
        outcome = random.choices(
            ["active", "purchased", "removed"], weights=[75, 15, 10], k=1
        )[0]

        # purchase_date: Optional, >= date_added, mutual exclusion with removal_date
        purchased = None
        if outcome == "purchased":
            if i % 42 == 0:
                # Purchased before added violation
                if isinstance(added, datetime):
                    purchased = added - timedelta(days=random.randint(1, 30))
                else:
                    purchased = fake.date_time_between(start_date="-3y", end_date="-2y")
            elif i % 52 == 0:
                purchased = fake.date_time_between(
                    start_date="+1m", end_date="+6m"
                )  # Future
            elif i % 72 == 0:
                # Purchased within 2 seconds of add (instrumentation artifact)
                if isinstance(added, datetime):
                    purchased = added + timedelta(seconds=random.randint(1, 2))
                else:
                    purchased = fake.date_time_between(start_date="-1y", end_date="now")
            else:
                # Valid: purchase_date >= date_added
                if isinstance(added, datetime):
                    purchased = fake.date_time_between(start_date=added, end_date="now")
                else:
                    purchased = fake.date_time_between(start_date="-1y", end_date="now")
        record["purchase_date"] = purchased

        # removal_date: Optional, >= date_added, mutual exclusion with purchase_date
        removed = None
        if outcome == "removed":
            if i % 44 == 0:
                # Removed before added violation
                if isinstance(added, datetime):
                    removed = added - timedelta(days=random.randint(1, 30))
                else:
                    removed = fake.date_time_between(start_date="-3y", end_date="-2y")
            elif i % 54 == 0:
                removed = fake.date_time_between(
                    start_date="+1m", end_date="+6m"
                )  # Future
            elif i % 74 == 0:
                # Removed within 1 second of add (bot/noise)
                if isinstance(added, datetime):
                    removed = added + timedelta(seconds=random.uniform(0.1, 1))
                else:
                    removed = fake.date_time_between(start_date="-1y", end_date="now")
            else:
                # Valid: removal_date >= date_added
                if isinstance(added, datetime):
                    removed = fake.date_time_between(start_date=added, end_date="now")
                else:
                    removed = fake.date_time_between(start_date="-1y", end_date="now")
        elif i % 81 == 0:
            # Both purchased and removed violation
            removed = fake.date_time_between(start_date="-6m", end_date="now")
            purchased = fake.date_time_between(start_date="-1y", end_date="-6m")
            record["purchase_date"] = purchased
        record["removal_date"] = removed

        # price_at_addition: Decimal >= 0, historical price
        if i % 49 == 0:
            price_added = round(random.uniform(-100, -1), 2)  # Negative violation
        elif i % 59 == 0:
            price_added = random.choice([0, 999999.99, 0.001])  # Extreme
        else:
            # Valid: based on product price with small variation
            price_added = round(base_price * random.uniform(0.9, 1.1), 2)
        record["price_at_addition"] = price_added

        # current_price: Decimal >= 0, should match products.retail_price
        if i % 63 == 0:
            # Price swing > 300% violation
            if isinstance(price_added, (int, float)) and price_added > 0:
                current_price = round(price_added * random.uniform(4, 6), 2)
            else:
                current_price = round(random.uniform(1000, 2000), 2)
        elif i % 73 == 0:
            # Massive drop < 25% violation
            if isinstance(price_added, (int, float)) and price_added > 0:
                current_price = round(price_added * 0.1, 2)
            else:
                current_price = round(random.uniform(1, 10), 2)
        elif i % 83 == 0:
            current_price = round(random.uniform(-50, -1), 2)  # Negative
        else:
            # Valid: close to product retail price
            current_price = round(base_price * random.uniform(0.8, 1.2), 2)
        record["current_price"] = current_price

        # priority: Canonical set [Low, Medium, High], default Medium
        if i % 32 == 0:
            priority = random.choice(
                [
                    "high",
                    "HIGH",
                    "H",
                    "1",
                    "medium",
                    "MEDIUM",
                    "M",
                    "2",
                    "low",
                    "LOW",
                    "L",
                    "3",
                ]
            )
        elif i % 62 == 0:
            priority = random.choice(["Hihg", "Mediun", "Loww"])  # Typos
        else:
            priority = random.choice(PRIORITY_CANONICAL)
        record["priority"] = priority

        # user_notes: 0-255 chars
        if i % 60 == 0:
            # Very long notes violation (> 255 chars)
            notes = fake.text(max_nb_chars=500)
        elif i % 80 == 0:
            notes = "<script>alert('xss')</script>"  # Script injection
        else:
            notes = random.choice(
                [
                    "Birthday gift idea",
                    "Wait for sale",
                    "Check reviews first",
                    "Alternative to consider",
                    "Must have!",
                    "Compare with other options",
                    "Gift for mom",
                    fake.sentence(nb_words=6),
                ]
            )
        record["user_notes"] = notes

        # list_name: Canonical set
        if i % 60 == 0:
            list_name = random.choice(
                ["wish list", "WISHLIST", "WishList"]
            )  # Case variations
        elif i % 70 == 0:
            # Too short (< 3 chars)
            list_name = random.choice(["AB", "X", "WL"])
        else:
            list_name = random.choice(LIST_NAME_CANONICAL)
        record["list_name"] = list_name

        # price_alert_enabled: Boolean, default FALSE
        if i % 44 == 0:
            alert_enabled = random.choice(["Y", "N", "Active", "Inactive"])
        else:
            alert_enabled = random.choice([True, False])
        record["price_alert_enabled"] = alert_enabled

        # notification_sent: Boolean, depends on price_alert_enabled
        if i % 45 == 0:
            notified = random.choice(
                ["Y", "N", "Yes", "No", "1", "0", "true", "false"]
            )
        elif i % 55 == 0:
            notified = random.choice(["Pending", "Sent", "Failed"])
        elif i % 75 == 0:
            # Notification sent but alert not enabled violation
            if alert_enabled == False:
                notified = True
            else:
                notified = False
        else:
            # Valid: notification_sent => price_alert_enabled
            if alert_enabled == True:
                notified = random.choice([True, False])
            else:
                notified = False
        record["notification_sent"] = notified

        # add_source: Canonical set
        if i % 55 == 0:
            source = random.choice(["android", "ios", "mobile"])  # Lowercase
        elif i % 65 == 0:
            # Both Mobile App and Android/iOS violation
            source = random.choice(["Mobile App, Android", "Mobile App/iOS"])
        else:
            source = random.choice(ADD_SOURCE_CANONICAL)
        record["add_source"] = source

        # desired_quantity: Integer >= 1, flag > 20
        if i % 65 == 0:
            qty = random.choice([-1, 0, -10])  # <= 0 violation
        elif i % 75 == 0:
            qty = random.choice([999, 0.5, 10000])  # Extreme
        elif i % 85 == 0:
            qty = random.randint(25, 100)  # > 20 (reseller flag)
        else:
            # Valid: 1-5 typical
            qty = random.randint(1, 5)
        record["desired_quantity"] = qty

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_wishlist_data(number_of_wishlists)
df = add_more_messiness(df)

output_file = "wishlists.xlsx"
df.to_excel(output_file, index=False)

In [18]:
for key, value in product_prices.items():
    print(f"{key}: {value}")

1000: 1564.36
1001: 618.37
1002: 29.03
1003: 160.56
1004: 518.82
1005: 73.44
1006: 17.91
1007: 103.17
1008: 201.6
1009: 1046.15
1010: 234.71
1011: 50.51
1012: 102.05
1013: 166.58
1014: 177.47
1015: 381.09
1016: 762.01
1017: 255.92
1018: 341.6
1019: 116.51
1020: 64.96
1021: 302.34
1022: 1387.34
1023: 324.89
1024: 211.08
1025: 84.63
1026: 615.24
1027: 1549.84
1028: 135.88
1029: 47.77
1030: 188.51
1031: 366.57
1032: 1475.31
1033: 78.9
1034: 93.67
1035: 41.66
1036: 101.93
1037: 184.39
1038: 560.69
1039: 294.38
1040: 95.15
1041: 299.02
1042: 153.42
1043: 526.67
1044: 1612.33
1045: 117.83
1046: 197.69
1047: 591.46
1048: 199.26
1049: 494.38
1050: 249.96
1051: 180.9
1052: 200.35
1054: 123.46
1055: 89.62
1056: 949.83
1057: 408.3
1058: 1558.89
1059: 128.5
1060: 103.11
1061: 62.95
1062: 180.36
1063: 334.8
1064: 61.48
1065: 132.13
1066: 266.1
1067: 260.74
1068: 733.65
1069: 111.03
1070: 710.39
1071: 13.92
1072: 310.2
1073: 518.73
1074: 179.64
1075: 1633.28
1076: 82.34
1077: 192.35
1078: 82.42
1079

### Shopping Cart Table Generator

In [19]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
STATUS_CANONICAL = ["Active", "Abandoned", "Converted (Ordered)", "Deleted"]
DEVICE_TYPE_CANONICAL = [
    "Desktop Web",
    "Mobile Web",
    "iOS App",
    "Android App",
    "Tablet Web",
]
PROMO_DISCOUNTS = {
    "SAVE10": 0.10,
    "WELCOME20": 0.20,
    "VIP15": 0.15,
    "LOYALTY":  0.10,
    "SUMMER2024": 0.25,
    "FLASH50": 0.50,
    "CLEARANCE": 0.60,
    "BLACKFRIDAY": 0.70,
}


def generate_messy_shopping_cart_data(
    num_rows=3000, customer_id_format="CUST", product_id_format="PROD"
):
    data = []
    used_cart_ids = []
    used_session_ids = []

    customer_ids = [
        f"CUST_{i + starting_customer_index}" for i in range(number_of_customers)
    ]
    product_ids = [
        f"PROD_{i + starting_product_index}" for i in range(number_of_products)
    ]

    cart_sessions = {}
    
    # -------------------------
    # Cart-level promo assignment (LOCAL to carts, not global)
    # -------------------------
    session_promos = {}  # session_id -> (coupon, discount_rate)

    for i in range(num_rows):
        record = {}

        # cart_item_id: Primary key, positive integer, unique
        if i % 53 == 0 and used_cart_ids:
            cart_id = random.choice(used_cart_ids)  # Duplicate violation
        else:
            cart_id = starting_cart_index + i
            used_cart_ids.append(cart_id)
        record["cart_item_id"] = cart_id

        # Decide guest vs registered
        is_guest = random.random() < 0.35  # 35% guest carts

        # customer_ref: Can be NULL for guest carts, FK to customers
        if is_guest:
            cust_id = None
        elif i % 59 == 0:
            cust_id = f"CUST_{99999}"  # FK violation
        else:
            cust_id = random.choice(customer_ids)
        record["customer_ref"] = cust_id

        # session_identifier: UUID format, required if customer_ref is NULL
        if is_guest and cust_id is None: 
            if i % 51 == 0 and used_session_ids: 
                session_id = random.choice(used_session_ids[-10:])  # Reuse session
            else:
                session_id = str(uuid.uuid4())
                used_session_ids.append(session_id)
        elif not is_guest and cust_id: 
            if random.random() < 0.7:
                session_id = str(uuid.uuid4())
                used_session_ids.append(session_id)
            else:
                session_id = str(uuid.uuid4())
                used_session_ids.append(session_id)
        else:
            session_id = str(uuid.uuid4())
            used_session_ids.append(session_id)
        record["session_identifier"] = session_id

        # -------------------------
        # Assign promo per SESSION (not per product globally)
        # -------------------------
        if session_id not in session_promos:
            if random.random() < 0.30:  # 30% of sessions have a promo
                coupon = random.choice(list(PROMO_DISCOUNTS.keys()))
                discount_rate = PROMO_DISCOUNTS[coupon]
                session_promos[session_id] = (coupon, discount_rate)
            else:
                session_promos[session_id] = (None, 0)

        # product_ref: Mandatory, FK to products
        if i % 57 == 0:
            prod_id = f"PROD_{9999}"  # FK violation
        else:
            if session_id and session_id in cart_sessions:
                if random.random() < 0.3 and cart_sessions[session_id]:
                    prod_id = random.choice(cart_sessions[session_id])
                else: 
                    prod_id = random.choice(product_ids)
                    cart_sessions[session_id].append(prod_id)
            else:
                prod_id = random.choice(product_ids)
                if session_id:
                    cart_sessions[session_id] = [prod_id]
        record["product_ref"] = prod_id

        # item_quantity: Integer >= 1, flag > 50 as suspicious
        if i % 42 == 0:
            quantity = random.choice([0, -1, -10])  # <= 0 violation
        elif i % 52 == 0:
            quantity = random.choice([999, 10000, -999])  # Extreme
        elif i % 62 == 0:
            quantity = random.choice([1.5, 2.3, 3.7])  # Decimal violation
        elif i % 72 == 0:
            quantity = random.randint(60, 200)  # > 50 suspicious
        else: 
            quantity = random.choices(
                [1, 2, 3, 4, 5, 10, 20], weights=[40, 25, 15, 10, 5, 3, 2], k=1
            )[0]
        record["item_quantity"] = quantity

        # price_per_unit
        price = product_prices.get(extract_product_number(prod_id), 0)
        record["price_per_unit"] = price

        # date_added_to_cart: <= now(), no future dates
        if i % 36 == 0:
            added_date = fake.date_time_between(start_date="-30d", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y"]
            added = added_date.strftime(random.choice(formats))
        elif i % 46 == 0:
            added = int(
                fake.date_time_between(start_date="-30d", end_date="now").timestamp()
            )
        elif i % 56 == 0:
            added = fake.date_time_between(
                start_date="+1d", end_date="+7d"
            )  # Future violation
        elif i % 66 == 0:
            added = fake.date_time_between(start_date="-1y", end_date="-6m")  # Very old
        else:
            added = fake.date_time_between(start_date="-30d", end_date="now")
        record["date_added_to_cart"] = added

        # status: Canonical set
        if i % 32 == 0:
            status = random.choice(
                ["active", "ACTIVE", "A", "1", "abandoned", "ABANDONED"]
            )
        elif i % 44 == 0:
            status = random.choice(
                ["Pending", "In Progress", "Expired", "Ordered"]
            )  # Non-canonical
        elif i % 54 == 0:
            status = random.choice(["Activ", "Abandond", "Convertd"])  # Typos
        else:
            if isinstance(added, datetime):
                days_old = (datetime.now() - added).days
                if days_old > 7:
                    status = random.choices(
                        ["Abandoned", "Converted (Ordered)", "Deleted"],
                        weights=[60, 35, 5],
                        k=1,
                    )[0]
                else: 
                    status = random.choices(
                        STATUS_CANONICAL, weights=[30, 45, 20, 5], k=1
                    )[0]
            else:
                status = random.choice(STATUS_CANONICAL)
        record["status"] = status

        # -------------------------
        # promo_code and discount:  Per SESSION (not polluting global state)
        # -------------------------
        if i % 60 == 0:
            # Messy data:  invalid promo code
            coupon = fake.text(max_nb_chars=100)  # Too long violation
            record["promo_code"] = coupon
            record["discount"] = 0
            discount_rate = 0
        else:
            # Get promo from session (LOCAL, not global)
            coupon, discount_rate = session_promos.get(session_id, (None, 0))
            record["promo_code"] = coupon
            record["discount"] = discount_rate

        # tax_amount: Decimal >= 0
        if i % 64 == 0:
            tax = round(random.uniform(-10, -1), 2)  # Negative violation
        elif i % 74 == 0:
            tax = random.choice([999, 0.001])  # Extreme
        else: 
            # Valid: tax = (price * qty - discount) * tax_rate
            if isinstance(price, (int, float)) and isinstance(quantity, (int, float)) and price > 0 and quantity > 0:
                subtotal = price * quantity
                discount_amount = subtotal * discount_rate
                taxable = max(0, subtotal - discount_amount)
                tax = round(taxable * product_tax_rate, 2)
            else:
                tax = 0
        record["tax_amount"] = tax

        # last_updated:  >= date_added_to_cart, <= now()
        if i % 58 == 0:
            if isinstance(added, datetime):
                updated = added - timedelta(hours=random.randint(1, 24))
            else:
                updated = fake.date_time_between(start_date="-35d", end_date="-31d")
        elif i % 69 == 0:
            updated = fake.date_time_between(start_date="+1d", end_date="+7d")
        else:
            if isinstance(added, datetime):
                updated = fake.date_time_between(start_date=added, end_date="now")
            else:
                updated = fake.date_time_between(start_date="-29d", end_date="now")
        record["last_updated"] = updated

        # device_type: Canonical set
        if i % 73 == 0:
            device = random.choice(
                ["ios", "Android", "Mobile", "Chrome", "Safari"]
            )  # Non-canonical
        else:
            device = random.choice(DEVICE_TYPE_CANONICAL)
        record["device_type"] = device

        # ip_address: Valid IPv4/IPv6
        if i % 56 == 0:
            ip = random.choice(
                [
                    "0.0.0.0",
                    "999.999.999.999",
                    "127.0.0.1",
                    "::1",
                ]
            )
        else:
            ip = fake.ipv4()
        record["ip_address"] = ip

        # saved_for_later:  Boolean
        if i % 67 == 0:
            saved = random.choice(
                ["Y", "N", "Yes", "No", "1", "0"]
            )  # String boolean
        elif i % 77 == 0:
            if status in ["Converted (Ordered)", "Converted"]:
                saved = True
            else: 
                saved = False
        elif i % 87 == 0:
            if cust_id is None:
                saved = True
            else:
                saved = False
        else:
            saved = random.choice([True, False])
        record["saved_for_later"] = saved

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[: 3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_shopping_cart_data(number_of_carts)
df = add_more_messiness(df)

output_file = "shopping_carts.xlsx"
df.to_excel(output_file, index=False)

Extracting product number from 'PROD_9999'
Extracting product number from 'PROD_1574'
Extracting product number from 'PROD_1828'
Extracting product number from 'PROD_1104'
Extracting product number from 'PROD_1080'
Extracting product number from 'PROD_1791'
Extracting product number from 'PROD_1379'
Extracting product number from 'PROD_1947'
Extracting product number from 'PROD_1410'
Extracting product number from 'PROD_1905'
Extracting product number from 'PROD_1269'
Extracting product number from 'PROD_1881'
Extracting product number from 'PROD_1994'
Extracting product number from 'PROD_1464'
Extracting product number from 'PROD_1156'
Extracting product number from 'PROD_1899'
Extracting product number from 'PROD_1080'
Extracting product number from 'PROD_1562'
Extracting product number from 'PROD_1205'
Extracting product number from 'PROD_1021'
Extracting product number from 'PROD_1234'
Extracting product number from 'PROD_1740'
Extracting product number from 'PROD_1362'
Extracting 

### Orders Items Table Generator

In [20]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
from collections import defaultdict

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
RETURN_STATUS_CANONICAL = [
    "None",
    "Return Requested",
    "Return Pending",
    "Returned",
    "Return Rejected",
    "Exchanged",
]
VALID_WAREHOUSES = [
    "WH-EAST-01",
    "WH-WEST-01",
    "WH-CENTRAL-01",
    "WH-NORTH-01",
    "WH-SOUTH-01",
    "DC-01",
    "DC-02",
    "STORE-001",
    "STORE-002",
    "DROPSHIP",
]

# Promo codes with discount rates
PROMO_DISCOUNTS = {
    "SAVE10":  0.10,
    "WELCOME20": 0.20,
    "VIP15":  0.15,
    "LOYALTY": 0.10,
    "SUMMER2024":  0.25,
    "FLASH50": 0.50,
    "CLEARANCE":  0.60,
    "BLACKFRIDAY": 0.70,
}


def generate_messy_order_items_data(
    num_rows=5000, order_id_format="ORD", product_id_format="PROD"
):
    data = []
    used_order_item_ids = []

    order_ids = [f"ORD_{i + starting_order_index}" for i in range(number_of_orders)]
    product_ids = [
        f"PROD_{i + starting_product_index}" for i in range(number_of_products)
    ]

    order_items_map = {}

    product_unit_weights = {
        pid: round(random.uniform(0.1, 10), 2) for pid in product_ids
    }

    # -------------------------
    # Assign coupons PER ORDER (not per product)
    # -------------------------
    for oid in order_ids:
        if random.random() < 0.30:  # 30% of orders have a coupon
            coupon = random.choice(list(PROMO_DISCOUNTS.keys()))
            order_coupons[oid] = coupon
            order_discount_rates[oid] = PROMO_DISCOUNTS[coupon]
        else:
            order_coupons[oid] = None
            order_discount_rates[oid] = 0

    for i in range(num_rows):
        record = {}

        # line_item_id:  Primary key, positive integer, unique
        if i % 53 == 0 and used_order_item_ids: 
            item_id = random.choice(used_order_item_ids)  # Duplicate violation
        else:
            item_id = starting_order_item_index + i
            used_order_item_ids.append(item_id)
        record["line_item_id"] = item_id

        # order_ref: Mandatory, format ^ORD_[0-9]+$
        if i % 61 == 0:
            order_id = f"ORD_{999999}"  # FK violation
        else:
            # Group items by order (realistic basket behavior)
            if random.random() < 0.6 and order_items_map:
                recent_orders = [
                    o for o, items in order_items_map.items() if len(items) < 10
                ]
                if recent_orders:
                    order_id = random.choice(recent_orders[-20:])
                else:
                    order_id = random.choice(order_ids)
            else:
                order_id = random.choice(order_ids)
        record["order_ref"] = order_id

        # product_ref: Mandatory, format ^PROD_[0-9]+$
        if i % 57 == 0:
            prod_id = f"PROD_{9999}"  # FK violation
        else: 
            if order_id in order_items_map and random.random() < 0.1:
                if order_items_map[order_id]:
                    prod_id = random.choice(order_items_map[order_id])
                else:
                    prod_id = random.choice(product_ids)
            else:
                popular_products = product_ids[: 20]
                prod_id = (
                    random.choice(popular_products)
                    if random.random() < 0.3
                    else random.choice(product_ids)
                )

        if order_id and prod_id:
            if order_id not in order_items_map:
                order_items_map[order_id] = []
            order_items_map[order_id].append(prod_id)
        record["product_ref"] = prod_id

        unit_weight = product_unit_weights.get(
            prod_id, round(random.uniform(0.1, 10), 2)
        )

        # qty_ordered: Integer, strictly > 0
        if i % 41 == 0:
            quantity = random.choice([0, -1, -10, -100])  # <= 0 violation
        elif i % 51 == 0:
            quantity = random.choice([999, 10000, 0.5, -999])  # Extreme/decimal
        elif i % 67 == 0:
            quantity = random.choice([1.5, 2.3, 3.7, 10.25])  # Decimal violation
        else:
            quantity = random.choices(
                [1, 2, 3, 4, 5, 10, 20, 50, 100],
                weights=[50, 20, 10, 5, 5, 5, 3, 1, 1],
                k=1,
            )[0]
        record["qty_ordered"] = quantity

        # is_gift: Boolean
        is_gift = False
        if i % 81 == 0:
            gift_val = random.choice(["Y", "N", "Yes", "No", "1", "0"])
        else:
            gift_val = random.choice([True, False])
            is_gift = gift_val == True
        record["is_gift"] = gift_val

        # unit_cost:  Decimal >= 0
        prod_num = extract_product_number(prod_id)
        product_cost = product_cost_prices.get(prod_num)
        record["unit_cost"] = product_cost

        # unit_selling_price: Decimal >= 0
        unit_price = product_prices.get(prod_num, 0)
        record["unit_selling_price"] = unit_price

        # -------------------------
        # DISCOUNT:  Per ORDER (not per product)
        # -------------------------
        discount_rate = order_discount_rates.get(order_id, 0)
        gross = max(0, unit_price * quantity) if isinstance(quantity, (int, float)) and isinstance(unit_price, (int, float)) else 0
        discount_val = min(
            round(gross * discount_rate, 2),
            gross
        )
        record["item_discount"] = discount_val

        # line_total: (unit_selling_price * qty_ordered) - item_discount
        if all(isinstance(x, (int, float)) for x in [quantity, unit_price]):
            total_price = round((quantity * unit_price) - discount_val, 2)
            if total_price < 0:
                total_price = 0
        else: 
            total_price = round(random.uniform(10, 1000), 2)
        record["line_total"] = total_price

        # profit_margin: line_total - (unit_cost * qty_ordered)
        if all(
            isinstance(x, (int, float))
            for x in [total_price, product_cost, quantity]
        ):
            margin = round(total_price - (product_cost * quantity), 2)
        else:
            margin = 0
        record["profit_margin"] = margin

        # tax_amount: Decimal >= 0, based on line_total
        tax = 0
        if isinstance(quantity, (int, float)) and isinstance(unit_price, (int, float)) and quantity > 0 and unit_price > 0:
            subtotal = unit_price * quantity
            taxable = max(0, subtotal - discount_val)
            tax = round(taxable * product_tax_rate, 2)
        record["tax_amount"] = tax

        # Store for order-level aggregation
        order_items_data[order_id].append({
            'discount': discount_val if isinstance(discount_val, (int, float)) else 0,
            'tax': tax if isinstance(tax, (int, float)) else 0,
            'line_total': total_price if isinstance(total_price, (int, float)) else 0
        })

        # product_sku
        if prod_id and prod_id.startswith("PROD_"):
            prod_num = prod_id.split("_")[1]
            sku = f"SKU-{prod_num}-{random.choice(['BLK', 'WHT', 'RED', 'BLU'])}"
        else:
            sku = f"SKU-{random.randint(1000, 9999)}-BLK"
        record["product_sku"] = sku

        # total_weight_kg
        if i % 82 == 0:
            weight = round(random.uniform(-5, -0.1), 2)  # Negative violation
        else:
            if isinstance(quantity, (int, float)) and quantity > 0:
                weight = round(unit_weight * quantity, 2)
            else:
                weight = round(random.uniform(0.1, 50), 2)
        record["total_weight_kg"] = weight

        # fulfillment_location
        warehouse = random.choice(VALID_WAREHOUSES)
        record["fulfillment_location"] = warehouse

        # return_status
        return_status = random.choice(RETURN_STATUS_CANONICAL)
        record["return_status"] = return_status

        # created_timestamp
        if i % 38 == 0:
            created = fake.date_time_between(
                start_date="-2y", end_date="now"
            ).strftime("%Y-%m-%d %H:%M:%S")
        elif i % 64 == 0:
            created = fake.date_time_between(
                start_date="+1d", end_date="+30d"
            )  # Future violation
        else:
            created = fake.date_time_between(start_date="-2y", end_date="now")
        record["created_timestamp"] = created

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols: 
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[: 3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_order_items_data(number_of_order_items)
df = add_more_messiness(df)

output_file = "order_items.xlsx"
df.to_excel(output_file, index=False)

Extracting product number from 'PROD_9999'
Extracting product number from 'PROD_1390'
Extracting product number from 'PROD_1928'
Extracting product number from 'PROD_1908'
Extracting product number from 'PROD_1008'
Extracting product number from 'PROD_1055'
Extracting product number from 'PROD_1234'
Extracting product number from 'PROD_1884'
Extracting product number from 'PROD_1651'
Extracting product number from 'PROD_1266'
Extracting product number from 'PROD_1488'
Extracting product number from 'PROD_1255'
Extracting product number from 'PROD_1931'
Extracting product number from 'PROD_1014'
Extracting product number from 'PROD_1928'
Extracting product number from 'PROD_1804'
Extracting product number from 'PROD_1012'
Extracting product number from 'PROD_1253'
Extracting product number from 'PROD_1984'
Extracting product number from 'PROD_1053'
Extracting product number from 'PROD_1248'
Extracting product number from 'PROD_1012'
Extracting product number from 'PROD_1007'
Extracting 

### Orders Table Generator

In [21]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker(["en_US", "en_GB", "fr_FR", "de_DE"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
STATUS_CANONICAL = [
    "Pending",
    "Processing",
    "Shipped",
    "Delivered",
    "Cancelled",
    "Returned",
]
CURRENCY_CANONICAL = ["USD", "EUR", "GBP", "CAD", "AUD"]
DEVICE_CANONICAL = ["Desktop", "Mobile", "Tablet"]
SHIPPING_METHOD_CANONICAL = [
    "Standard",
    "Priority",
    "Express",
    "2-Day",
    "White Glove",
    "Cash on Delivery",
]
PAYMENT_METHOD_CANONICAL = [
    "Credit Card",
    "PayPal",
    "Apple Pay",
    "Bitcoin",
    "Gift Card",
    "Cash on Delivery",
]
MARKETING_CHANNEL_CANONICAL = [
    "Organic Search",
    "Paid Search",
    "Social Media",
    "Direct",
    "Email",
    "Referral",
    "Display Ads",
]


def generate_messy_orders_data(num_rows=2500, customer_id_format="CUST"):
    data = []
    used_order_ids = []

    customer_ids = [
        f"CUST_{i + starting_customer_index}" for i in range(number_of_customers)
    ]

    for i in range(num_rows):
        record = {}

        # order_ref: Primary key, positive integer, unique
        if i % 53 == 0 and used_order_ids:
            order_id = random.choice(used_order_ids)  # Duplicate violation
        else:
            order_id = starting_order_index + i
            used_order_ids.append(order_id)
        order_id = f"ORD_{order_id}"
        record["order_ref"] = order_id

        # customer_ref: Mandatory, format ^CUST_[1-9][0-9]*$
        if i % 61 == 0:
            cust_id = f"CUST_{99999}"  # FK violation
        else:
            cust_id = random.choice(customer_ids)
        record["customer_ref"] = cust_id

        # order_status: Canonical set with date dependencies
        if i % 37 == 0:
            status = random.choice(
                ["pending", "PENDING", "P", "1", "shipped", "SHIPPED"]
            )
        elif i % 47 == 0:
            status = random.choice(["In Transit", "Complete", "Failed", "On Hold"])
        elif i % 57 == 0:
            status = random.choice(
                ["Pendng", "Proccessing", "Shiped", "Deliverd", "Cancled"]
            )
        else:
            status = random.choices(
                STATUS_CANONICAL, weights=[10, 15, 20, 40, 10, 5], k=1
            )[0]
        record["order_status"] = status

        # purchase_date: <= now()
        if i % 31 == 0:
            order_date_dt = fake.date_time_between(start_date="-2y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y", "%Y%m%d"]
            order_date = order_date_dt.strftime(random.choice(formats))
        elif i % 41 == 0:
            order_date = int(
                fake.date_time_between(start_date="-2y", end_date="now").timestamp()
            )
        elif i % 51 == 0:
            order_date = fake.date_time_between(
                start_date="+1d", end_date="+30d"
            )  # Future violation
        elif i % 67 == 0:
            order_date = fake.date_time_between(
                start_date="-10y", end_date="-5y"
            )  # Very old
        else:
            order_date = fake.date_time_between(start_date="-2y", end_date="now")
        record["purchase_date"] = order_date

        # shipment_date: Required if Shipped/Delivered/Returned, >= purchase_date
        shipped_date = None
        requires_shipment = status in ["Shipped", "Delivered", "Returned"]

        if requires_shipment:
            if i % 42 == 0: # Violation: shipped before ordered
                if isinstance(order_date, datetime):
                    shipped_date = order_date - timedelta(days=random.randint(1, 5))
                else:
                    shipped_date = fake.date_time_between(
                        start_date="-3y", end_date="-2y"
                    )
            elif i % 52 == 0:
                if isinstance(order_date, datetime):
                    shipped_date = (
                        order_date + timedelta(days=random.randint(1, 3))
                    ).strftime("%Y-%m-%d")
                else:
                    shipped_date = fake.date_between(start_date="-1y", end_date="today")
            elif i % 62 == 0:
                shipped_date = fake.date_time_between(
                    start_date="+1d", end_date="+30d"
                )  # Future
            else:
                # Valid: shipment_date >= purchase_date
                if isinstance(order_date, datetime):
                    max_ship = min(order_date + timedelta(days=7), datetime.now())
                    shipped_date = fake.date_time_between(
                        start_date=order_date, end_date=max_ship
                    )
                else:
                    shipped_date = fake.date_time_between(
                        start_date="-1y", end_date="now"
                    )
        elif i % 71 == 0: # Violation: not shipped but has date
            shipped_date = fake.date_time_between(start_date="-1y", end_date="now")
        record["shipment_date"] = shipped_date

        # delivery_date: Required if Delivered/Returned, >= shipment_date
        delivered_date = None
        requires_delivery = status in ["Delivered", "Returned"]

        if requires_delivery:
            if i % 46 == 0: # Violation: delivered before shipped
                if isinstance(shipped_date, datetime):
                    delivered_date = shipped_date - timedelta(days=random.randint(1, 3))
                elif isinstance(order_date, datetime):
                    delivered_date = order_date - timedelta(days=random.randint(1, 5))
                else:
                    delivered_date = fake.date_time_between(
                        start_date="-3y", end_date="-2y"
                    )
            elif i % 56 == 0:
                delivered_date = shipped_date  # Same day
            elif i % 66 == 0:
                delivered_date = fake.date_time_between(
                    start_date="+1d", end_date="+30d"
                )  # Future
            else:
                # Valid: delivery_date >= shipment_date
                if isinstance(shipped_date, datetime):
                    max_deliver = min(shipped_date + timedelta(days=10), datetime.now())
                    delivered_date = fake.date_time_between(
                        start_date=shipped_date, end_date=max_deliver
                    )
                elif isinstance(order_date, datetime):
                    max_deliver = min(order_date + timedelta(days=14), datetime.now())
                    delivered_date = fake.date_time_between(
                        start_date=order_date + timedelta(days=3), end_date=max_deliver
                    )
                else:
                    delivered_date = fake.date_time_between(
                        start_date="-6m", end_date="now"
                    )
        elif i % 76 == 0: # Violation: not delivered but has date
            delivered_date = fake.date_time_between(start_date="-1y", end_date="now")
        record["delivery_date"] = delivered_date

        # order_subtotal: Decimal >= 0
        # if i % 44 == 0:
        #     subtotal = round(random.uniform(-500, -10), 2)  # Negative violation
        # elif i % 54 == 0:
        #     subtotal = 0  # Zero subtotal
        # elif i % 64 == 0:
        #     subtotal = random.choice([999999.99, 0.001, -9999])
        # else:
        #     subtotal = round(random.uniform(10, 2000), 2)
        subtotal = 0
        if order_id in order_items_data:
            subtotal = sum(item['line_total'] for item in order_items_data[order_id])
            subtotal = round(subtotal, 2)
        record["order_subtotal"] = subtotal

        # tax_total: Decimal >= 0, typically 0-25% of subtotal
        # if i % 48 == 0:
        #     tax = round(random.uniform(-50, -1), 2)  # Negative violation
        # elif i % 58 == 0: # Tax > subtotal violation
        #     if isinstance(subtotal, (int, float)) and subtotal > 0:
        #         tax = subtotal * 1.5
        #     else:
        #         tax = 999
        # else:
        #     # Valid: tax 5-15% of subtotal
        #     if isinstance(subtotal, (int, float)) and subtotal > 0:
        #         tax = round(subtotal * random.uniform(0.05, 0.15), 2)
        #     else:
        #         tax = 0
        tax = 0
        if order_id in order_items_data:
            tax = sum(item['tax'] for item in order_items_data[order_id])
            tax = round(tax, 2)
        record["tax_total"] = tax

        # shipping_fee: Decimal >= 0, depends on shipping_method
        if i % 49 == 0:
            shipping = round(random.uniform(-20, -1), 2)  # Negative violation
        elif i % 59 == 0:
            shipping = random.choice([999, 0.001, -99])
        else:
            # Valid: free for large orders, otherwise 5-50
            if isinstance(subtotal, (int, float)) and subtotal > 100:
                shipping = (
                    0 if random.random() < 0.3 else round(random.uniform(5, 25), 2)
                )
            else:
                shipping = round(random.uniform(5, 50), 2)
        record["shipping_fee"] = shipping

        # discount_total: Decimal >= 0, <= subtotal + shipping + tax
        # if i % 55 == 0:
        #     discount = round(random.uniform(-100, -10), 2)  # Negative violation
        # elif i % 65 == 0: # Discount > subtotal violation
        #     if isinstance(subtotal, (int, float)) and subtotal > 0:
        #         discount = subtotal * 1.2
        #     else:
        #         discount = 1000
        # else:
        #     # Valid: discount <= subtotal
        #     if isinstance(subtotal, (int, float)) and subtotal > 0:
        #         discount = (
        #             round(subtotal * random.uniform(0, 0.3), 2)
        #             if random.random() < 0.4
        #             else 0
        #         )
        #     else:
        #         discount = 0
        discount = 0
        if order_id in order_items_data:
            discount = sum(item['discount'] for item in order_items_data[order_id])
            discount = round(discount, 2)
        record["discount_total"] = discount

        # grand_total: = subtotal + tax + shipping - discount, >= 0
        # if i % 46 == 0: # Wrong calculation violation
        #     if all(
        #         isinstance(x, (int, float)) for x in [subtotal, tax, shipping, discount]
        #     ):
        #         correct_total = subtotal + tax + shipping - discount
        #         total = correct_total * random.uniform(0.5, 1.5)
        #     else:
        #         total = random.uniform(10, 1000)
        # elif i % 56 == 0:
        #     total = round(random.uniform(-500, -1), 2)  # Negative violation
        # elif i % 66 == 0:
        #     total = 0
        # else:
            # Valid: grand_total = subtotal + tax + shipping - discount
        if all(
            isinstance(x, (int, float)) for x in [subtotal, tax, shipping, discount]
        ):
            total = round(subtotal + tax + shipping, 2)
            if total < 0:
                total = 0  # Domain >= 0
        elif isinstance(subtotal, (int, float)):
            total = round(subtotal * 1.1, 2)
        else:
            total = round(random.uniform(10, 2000), 2)
        record["grand_total"] = total

        # currency_code: ISO 4217 3-letter uppercase
        if i % 42 == 0:
            currency = random.choice(["US", "EURO", "Dollar", "$", "€"])
        elif i % 52 == 0:
            currency = random.choice(["JPY", "CNY", "INR", "BTC", "DOGE"])
        else:
            currency = random.choices(
                CURRENCY_CANONICAL, weights=[60, 20, 10, 5, 5], k=1
            )[0]
        record["currency_code"] = currency

        # shipping_method: Canonical set
        ship_method = random.choice(SHIPPING_METHOD_CANONICAL)
        # Cross-field: Free Shipping should have shipping_fee = 0
        if (
            ship_method == "Standard"
            and isinstance(shipping, (int, float))
            and shipping == 0
        ):
            ship_method = "Free Shipping"  # Map correctly
        record["shipping_method"] = ship_method

        # device_category: Canonical set
        if i % 40 == 0:
            device = random.choice(["mobile", "MOBILE", "desk", "DESKTOP"])
        else:
            device = random.choices(DEVICE_CANONICAL, weights=[40, 50, 10], k=1)[0]
        record["device_category"] = device

        # payment_method: Canonical set
        payment = random.choice(PAYMENT_METHOD_CANONICAL)
        record["payment_method"] = payment

        # marketing_channel: Canonical set
        if i % 35 == 0:
            channel = random.choice(
                ["google", "GOOGLE", "fb", "FB", "email", "EMAIL"]
            )
        else:
            channel = random.choice(MARKETING_CHANNEL_CANONICAL)
        record["marketing_channel"] = channel

        # coupon_code: If present, discount_total > 0
        # coupon = random.choice(
        #     [
        #         "SAVE10",
        #         "WELCOME20",
        #         "FREESHIP",
        #         "SUMMER2024",
        #         "VIP15",
        #         "FLASH50",
        #         "BLACKFRIDAY",
        #         "LOYALTY",
        #     ]
        # )
        # record["coupon_code"] = coupon
        coupon = order_coupons.get(order_id, None)  # Get actual coupon assigned
        record["coupon_code"] = coupon

        # item_count: Integer > 0
        if i % 65 == 0:
            items = random.choice([0, -1, -5])  # <= 0 violation
        elif i % 75 == 0:
            items = random.choice([999, 0.5, 10000])
        else:
            items = random.choices(
                [1, 2, 3, 4, 5, 10, 20], weights=[30, 25, 20, 10, 10, 3, 2], k=1
            )[0]
        record["item_count"] = items

        # customer_ip: Valid IPv4/IPv6
        if i % 65 == 0:
            ip = random.choice(["0.0.0.0", "999.999.999.999"])
        else:
            ip = fake.ipv4()
        record["customer_ip"] = ip

        # refund_amount: >= 0, <= grand_total, only if Cancelled/Returned
        is_refundable = status in ["Returned", "Cancelled"]
        if is_refundable:
            if i % 80 == 0: # Refund > total violation
                if isinstance(total, (int, float)) and total > 0:
                    refund = total * 1.5
                else:
                    refund = 1000
            else:
                # Valid: refund <= grand_total
                if isinstance(total, (int, float)) and total > 0:
                    refund = round(total * random.uniform(0.5, 1.0), 2)
                else:
                    refund = round(random.uniform(10, 500), 2)
            record["refund_amount"] = refund

        # record_created: <= now(), close to purchase_date
        if i % 40 == 0:
            created = fake.date_time_between(
                start_date="-2y", end_date="now"
            ).strftime("%Y-%m-%d %H:%M:%S")
        elif i % 50 == 0: # Violation: created after order
            if isinstance(order_date, datetime):
                created = order_date + timedelta(days=random.randint(1, 30))
            else:
                created = fake.date_time_between(start_date="+1d", end_date="+30d")
        else:
            # Valid: close to purchase_date
            if isinstance(order_date, datetime):
                created = order_date + timedelta(minutes=random.randint(0, 60))
            else:
                created = fake.date_time_between(start_date="-2y", end_date="now")
        record["record_created"] = created

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_orders_data(number_of_orders)
df = add_more_messiness(df)

output_file = "orders.xlsx"
df.to_excel(output_file, index=False)

### Payments Table Generator

In [22]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
PAYMENT_TYPE_CANONICAL = [
    "Credit Card",
    "Debit Card",
    "PayPal",
    "Apple Pay",
    "Google Pay",
    "Bank Transfer",
    "Gift Card",
    "Cash on Delivery",
    "Buy Now Pay Later",
]
GATEWAY_PROVIDER_CANONICAL = [
    "Stripe",
    "Square",
    "PayPal",
    "Authorize.net",
    "Adyen",
    "Braintree",
    "Plaid",
    "COD",
]
PAYMENT_STATUS_CANONICAL = [
    "Pending",
    "Completed",
    "Failed",
    "Refunded",
    "Partially Refunded",
    "Cancelled",
]
CARD_BRAND_CANONICAL = ["Visa", "MasterCard", "American Express", "Discover", "JCB"]
CURRENCY_CANONICAL = ["USD", "EUR", "GBP", "CAD", "AUD"]

# Payment type to provider mapping
PAYMENT_PROVIDER_MAP = {
    "Credit Card": ["Stripe", "Square", "Authorize.net", "Braintree", "Adyen"],
    "Debit Card": ["Stripe", "Square", "Authorize.net", "Adyen"],
    "PayPal": ["PayPal"],
    "Apple Pay": ["Stripe", "Square", "Adyen", "Braintree"],
    "Google Pay": ["Stripe", "Square", "Adyen", "Braintree"],
    "Bank Transfer": ["Plaid"],
    "Gift Card": ["Stripe", "Square"],  # Internal handling
    "Cash on Delivery": ["COD"],
    "Buy Now Pay Later": ["Stripe", "Adyen"],  # Klarna/Afterpay through these
}


def generate_gateway_transaction_id(provider, payment_status):
    """Generate provider-specific transaction ID format"""
    if provider == "Stripe":
        return f"pi_{uuid.uuid4().hex[:24]}"
    elif provider == "PayPal":
        return f"PP-{uuid.uuid4().hex[:20].upper()}"
    elif provider == "Square":
        return f"sq_{uuid.uuid4().hex[:22]}"
    elif provider == "Adyen":
        return f"ADY-{uuid.uuid4().hex[:16].upper()}"
    elif provider == "Braintree":
        return f"bt_{uuid.uuid4().hex[:20]}"
    elif provider == "Authorize.net":
        return f"AUTH-{random.randint(100000000, 999999999)}"
    elif provider == "Plaid":
        return f"plaid_{uuid.uuid4().hex[:18]}"
    elif provider == "COD":
        return f"COD-{uuid.uuid4().hex[:16].upper()}"
    else:
        return f"TXN-{uuid.uuid4().hex[:16].upper()}"


def generate_messy_payments_data(num_rows=3500, order_id_format="ORD"):
    data = []
    used_payment_ids = []
    used_transaction_ids = []

    order_ids = [f"ORD_{i + starting_order_index}" for i in range(number_of_orders)]

    # Generate order amounts for consistency
    order_amounts = {oid: round(random.uniform(10, 2000), 2) for oid in order_ids}

    for i in range(num_rows):
        record = {}

        # payment_ref: Primary key, positive integer, unique
        if i % 53 == 0 and used_payment_ids:
            payment_id = random.choice(used_payment_ids)  # Duplicate violation
        else:
            payment_id = starting_payment_index + i
            used_payment_ids.append(payment_id)
        record["payment_ref"] = payment_id

        # order_ref: Mandatory, FK to orders
        if i % 61 == 0:
            order_id = f"ORD_{999999}"  # FK violation
        else:
            order_id = random.choice(order_ids)
        record["order_ref"] = order_id

        # Get expected amount for this order
        expected_amount = order_amounts.get(
            order_id, round(random.uniform(10, 2000), 2)
        )

        # payment_type: Canonical set
        if i % 37 == 0:
            method = random.choice(["CC", "credit card", "CREDIT_CARD", "Card", "Visa"])
        elif i % 57 == 0:
            method = random.choice(
                ["Credt Card", "PayPall", "Banck Transfer", "Appel Pay"]
            )
        else:
            method = random.choice(PAYMENT_TYPE_CANONICAL)
        record["payment_type"] = method

        # gateway_provider: Must match payment_type
        if i % 39 == 0:
            # Mismatched provider violation
            if method == "PayPal":
                provider = "Stripe"
            elif method == "Credit Card":
                provider = "PayPal"
            else:
                provider = "Stripe"
        elif i % 59 == 0:
            provider = random.choice(["Strpe", "Sqaure", "PayPl", "Klarrna"])
        else:
            # Valid: provider matches payment_type
            if method in PAYMENT_PROVIDER_MAP:
                provider = random.choice(PAYMENT_PROVIDER_MAP[method])
            else:
                provider = random.choice(GATEWAY_PROVIDER_CANONICAL[:5])
        record["gateway_provider"] = provider

        # payment_status: Canonical set
        if i % 33 == 0:
            status = random.choice(
                ["completed", "COMPLETED", "Complete", "1", "Success"]
            )
        elif i % 44 == 0:
            status = random.choice(
                ["In Progress", "Processing", "Approved", "Declined"]
            )
        elif i % 54 == 0:
            status = random.choice(["Complted", "Pendng", "Faild", "Refnded"])
        else:
            status = random.choices(
                PAYMENT_STATUS_CANONICAL, weights=[10, 60, 10, 10, 5, 5], k=1
            )[0]
        record["payment_status"] = status

        # transaction_date: <= now()
        if i % 36 == 0:
            payment_date_dt = fake.date_time_between(start_date="-1y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y"]
            payment_date = payment_date_dt.strftime(random.choice(formats))
        elif i % 46 == 0:
            payment_date = int(
                fake.date_time_between(start_date="-1y", end_date="now").timestamp()
            )
        elif i % 56 == 0:
            payment_date = fake.date_time_between(
                start_date="+1d", end_date="+30d"
            )  # Future violation
        else:
            payment_date = fake.date_time_between(start_date="-1y", end_date="now")
        record["transaction_date"] = payment_date

        # gateway_transaction_id: Required for Completed, provider-specific format
        if i % 51 == 0 and used_transaction_ids:
            trans_id = random.choice(used_transaction_ids)  # Duplicate violation
        elif i % 67 == 0:
            # Wrong format for provider
            trans_id = f"WRONG-{random.randint(1000, 9999)}"
        else:
            # Valid: provider-specific format
            trans_id = generate_gateway_transaction_id(provider, status)
            used_transaction_ids.append(trans_id)
        record["gateway_transaction_id"] = trans_id

        # payment_amount: > 0 for Completed
        if i % 45 == 0:
            amount = round(random.uniform(-500, -10), 2)  # Negative violation
        elif i % 55 == 0:
            amount = 0  # Zero violation
        elif i % 65 == 0:
            amount = round(expected_amount * random.uniform(0.5, 1.5), 2)  # Mismatch
        elif i % 75 == 0:
            amount = random.choice([999999.99, 0.01, -9999])  # Extreme
        else:
            # Valid: matches order amount
            if status in ["Completed", "Refunded", "Partially Refunded"]:
                amount = expected_amount
            elif status == "Failed":
                amount = expected_amount  # Failed attempts record attempted amount
            else:
                amount = expected_amount
        record["payment_amount"] = amount

        # transaction_fee: >= 0, < payment_amount, typically 1-5%
        if i % 48 == 0:
            fee = round(random.uniform(-10, -1), 2)  # Negative violation
        elif i % 58 == 0:
            fee = random.choice([999, 0.001, -99])  # Extreme
        elif i % 68 == 0:
            # Fee > 10% violation
            if isinstance(amount, (int, float)) and amount > 0:
                fee = round(amount * 0.15, 2)
            else:
                fee = 50
        else:
            # Valid: fee based on payment type
            if isinstance(amount, (int, float)) and amount > 0:
                if method in ["Cash on Delivery"]:
                    fee = 0  # COD has no transaction fee
                elif method in ["Gift Card"]:
                    fee = 0  # Gift card typically no fee
                elif method in ["Credit Card", "Debit Card"]:
                    fee = round(amount * 0.029 + 0.30, 2)  # 2.9% + $0.30
                elif method == "PayPal":
                    fee = round(amount * 0.0349 + 0.49, 2)  # 3.49% + $0.49
                elif method == "Bank Transfer":
                    fee = round(amount * 0.008, 2)  # 0.8%
                else:
                    fee = round(amount * 0.025, 2)  # 2.5%
            else:
                fee = round(random.uniform(0.30, 50), 2)
        record["transaction_fee"] = fee

        # refund_total: >= 0, <= payment_amount, consistent with status
        if i % 86 == 0:
            # Not refunded but has refund amount violation
            if status not in ["Refunded", "Partially Refunded"]:
                refund_amount = round(random.uniform(10, 500), 2)
            else:
                refund_amount = 0
        elif i % 80 == 0:
            # Refund > payment violation
            if isinstance(amount, (int, float)) and amount > 0:
                refund_amount = amount * 1.5
            else:
                refund_amount = 1000
        elif i % 90 == 0:
            refund_amount = round(random.uniform(-100, -10), 2)  # Negative violation
        else:
            # Valid: consistent with status
            if status == "Refunded":
                refund_amount = amount if isinstance(amount, (int, float)) else 0
            elif status == "Partially Refunded":
                if isinstance(amount, (int, float)) and amount > 0:
                    refund_amount = round(amount * random.uniform(0.1, 0.9), 2)
                else:
                    refund_amount = round(random.uniform(10, 200), 2)
            else:
                refund_amount = 0  # No refund for other statuses
        record["refund_total"] = refund_amount

        # refund_processed_date: Required if Refunded/Partially Refunded, >= transaction_date
        refund_date = None
        has_refund = isinstance(refund_amount, (int, float)) and refund_amount > 0

        if has_refund:
            if i % 52 == 0:
                # Refund before payment violation
                if isinstance(payment_date, datetime):
                    refund_date = payment_date - timedelta(days=random.randint(1, 10))
                else:
                    refund_date = fake.date_time_between(
                        start_date="-2y", end_date="-1y"
                    )
            elif i % 62 == 0:
                refund_date = fake.date_between(start_date="-6m", end_date="today")
            elif i % 72 == 0:
                refund_date = fake.date_time_between(
                    start_date="+1d", end_date="+30d"
                )  # Future
            else:
                # Valid: refund_date >= transaction_date
                if isinstance(payment_date, datetime):
                    max_refund = min(payment_date + timedelta(days=90), datetime.now())
                    refund_date = fake.date_time_between(
                        start_date=payment_date, end_date=max_refund
                    )
                else:
                    refund_date = fake.date_time_between(
                        start_date="-6m", end_date="now"
                    )
        record["refund_processed_date"] = refund_date

        # currency_code: ISO 4217
        if i % 42 == 0:
            currency = random.choice(["US", "EURO", "Dollar", "$"])
        else:
            currency = random.choices(
                CURRENCY_CANONICAL, weights=[70, 15, 5, 5, 5], k=1
            )[0]
        record["currency_code"] = currency

        # card_last_four: Only for card payments, exactly 4 digits
        is_card_payment = method in ["Credit Card", "Debit Card", "CC", "credit card"]
        if is_card_payment:
            if i % 60 == 0:
                card_last4 = random.choice(["XXXX", "****", "0000"])  # Invalid
            elif i % 70 == 0:
                card_last4 = random.choice(["123", "12345", "1"])  # Wrong length
            else:
                card_last4 = str(random.randint(1000, 9999))
            record["card_last_four"] = card_last4

        # card_brand: Only for card payments, canonical set
        if is_card_payment:
            brand = random.choice(CARD_BRAND_CANONICAL)
            record["card_brand"] = brand
        elif method in ["Bank Transfer", "PayPal", "Gift Card", "Cash on Delivery"]:
            # Non-card payments should have NULL card fields
            if i % 85 == 0:
                record["card_last_four"] = str(random.randint(1000, 9999))  # Violation
                record["card_brand"] = random.choice(CARD_BRAND_CANONICAL)

        # authorization_code: Required for Completed card payments
        if status in ["Completed", "completed", "COMPLETED"] and is_card_payment:
            auth_code = f"AUTH-{uuid.uuid4().hex[:12].upper()}"
            record["authorization_code"] = auth_code
        elif status in ["Failed", "Cancelled"] and i % 83 == 0:
            # Auth code for failed payment violation
            record["authorization_code"] = f"AUTH-{uuid.uuid4().hex[:12].upper()}"

        # risk_score: 0-100
        if i % 85 == 0:
            risk = random.choice([-10, 150, 999])  # Out of range
        elif i % 91 == 0:
            # High risk but Completed violation
            if status == "Completed":
                risk = random.randint(80, 99)
            else:
                risk = random.randint(1, 30)
        else:
            # Valid: higher risk for failed payments
            if status in ["Failed", "failed", "Faild"]:
                risk = random.randint(50, 99)
            elif status in ["Completed", "completed"]:
                risk = random.randint(1, 40)
            else:
                risk = random.randint(1, 60)
        record["risk_score"] = risk

        # customer_ip: Valid IPv4/IPv6
        if i % 80 == 0:
            ip = random.choice(["0.0.0.0", "999.999.999.999", "127.0.0.1"])
        else:
            ip = fake.ipv4()
        record["customer_ip"] = ip

        # billing_country: ISO 3166-1 alpha-2
        if i % 85 == 0:
            country = random.choice(
                ["USA", "United States", "UK"]
            )  # Invalid
        else:
            country = fake.country_code()
        record["billing_country"] = country

        # retry_attempt: Integer >= 0
        if i % 78 == 0:
            retry = random.randint(-5, -1)  # Negative violation
        elif i % 88 == 0:
            retry = random.randint(10, 20)  # Unusually high
        else:
            # Valid: 0 for first attempt
            if status in ["Failed", "Cancelled"]:
                retry = random.randint(0, 3)
            else:
                retry = 0
        record["retry_attempt"] = retry

        # record_created: >= transaction_date
        if i % 40 == 0:
            created = fake.date_time_between(
                start_date="-1y", end_date="now"
            ).strftime("%Y-%m-%d %H:%M:%S")
        elif i % 50 == 0:
            # Created before payment violation
            if isinstance(payment_date, datetime):
                created = payment_date - timedelta(days=random.randint(1, 30))
            else:
                created = fake.date_time_between(start_date="-2y", end_date="-1y")
        else:
            # Valid: close to transaction_date
            if isinstance(payment_date, datetime):
                created = payment_date + timedelta(minutes=random.randint(0, 60))
            else:
                created = fake.date_time_between(start_date="-1y", end_date="now")
        record["record_created"] = created

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_payments_data(number_of_payments)
df = add_more_messiness(df)

output_file = "payments.xlsx"
df.to_excel(output_file, index=False)

### Inventory Table Generator

In [23]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

VALID_WAREHOUSES = [
    "WH-EAST-01",
    "WH-WEST-01",
    "WH-CENTRAL-01",
    "WH-NORTH-01",
    "WH-SOUTH-01",
    "DC-01",
    "DC-02",
    "STORE-001",
    "STORE-002",
    "DROPSHIP",
]
STOCK_STATUS_CANONICAL = ["In Stock", "Low Stock", "Out of Stock", "Discontinued"]

starting_inventory_index = 10000
starting_product_index = 1
starting_supplier_index = 1
number_of_suppliers = 50
number_of_inventories = 1500


def derive_stock_status(available_qty, min_stock_level, is_discontinued=False, is_expired=False):
    """Derive stock_status from available_qty and min_stock_level per exact rules."""
    if available_qty == 0:
        return "Out of Stock"
    if is_discontinued:
        return "Discontinued"
    if is_expired:
        return "Out of Stock"
    if available_qty is None or min_stock_level is None: 
        return "Out of Stock"
    if not isinstance(available_qty, (int, float)) or not isinstance(min_stock_level, (int, float)):
        return "Out of Stock"
    if 0 < available_qty < min_stock_level:
        return "Low Stock"
    if available_qty >= min_stock_level:
        return "In Stock"
    return "Out of Stock"


def generate_messy_inventory_data(
    num_rows=1500, product_id_format="PROD", supplier_id_format="SUPP"
):
    data = []
    used_inventory_ids = []

    product_ids = [f"PROD_{i + starting_product_index}" for i in range(num_rows)]
    supplier_ids = [
        f"SUPP_{i + starting_supplier_index}" for i in range(number_of_suppliers)
    ]

    product_inventory_map = {}
    product_categories = {
        "high_turnover": product_ids[: int(len(product_ids) * 0.2)],
        "medium_turnover": product_ids[
            int(len(product_ids) * 0.2) : int(len(product_ids) * 0.6)
        ],
        "low_turnover": product_ids[
            int(len(product_ids) * 0.6) : int(len(product_ids) * 0.9)
        ],
        "obsolete": product_ids[int(len(product_ids) * 0.9) :],
    }

    perishable_products = set(random.sample(product_ids, int(len(product_ids) * 0.3)))

    for i in range(num_rows):
        record = {}

        # inv_id: Primary key, positive integer, unique (with intentional duplicates at 2%)
        if i % 50 == 0 and used_inventory_ids:
            inv_id = random.choice(used_inventory_ids)
        else:
            inv_id = starting_inventory_index + i
            used_inventory_ids.append(inv_id)
        record["inv_id"] = inv_id

        # product_ref: Mandatory, ^PROD_[0-9]+$ format, uppercase
        if i % 60 == 0:
            prod_id = "PROD_9999"
        elif i % 70 == 0 and product_inventory_map:
            prod_id = random.choice(list(product_inventory_map.keys())[: 10])
        else:
            available = [p for p in product_ids if p not in product_inventory_map]
            prod_id = (
                random.choice(available) if available else random.choice(product_ids)
            )

        product_inventory_map[prod_id] = True
        record["product_ref"] = prod_id.upper()

        product_category = "medium_turnover"
        for cat, prods in product_categories.items():
            if prod_id in prods:
                product_category = cat
                break

        is_discontinued = product_category == "obsolete"
        is_perishable = prod_id in perishable_products

        # vendor_id: Mandatory, ^SUPP_[1-9][0-9]*$ format, no SUPP_0, uppercase
        if i % 35 == 0:
            supp_id = "SUPP_999"
        else:
            supp_id = random.choice(supplier_ids)
        record["vendor_id"] = supp_id.upper()

        # created_date: Must be <= today, YYYY-MM-DD format (with violations)
        if i % 50 == 0:
            created = fake.date_time_between(start_date="+1d", end_date="+30d").strftime("%Y-%m-%d")
        else:
            created = fake.date_time_between(start_date="-2y", end_date="now").strftime("%Y-%m-%d")
        record["created_date"] = created
        try:
            created_dt = datetime.strptime(created, "%Y-%m-%d")
        except:
            created_dt = datetime.now()

        # expiry_date: For perishables only, >= created_date, >= last_restock_date (with violations)
        is_expired = False
        if is_perishable:
            if i % 80 == 0:
                expiry = fake.date_between(start_date="-30d", end_date="-1d")
                is_expired = True
            else: 
                base_date = created_dt.date()
                expiry = fake.date_between(start_date=base_date, end_date="+2y")
                is_expired = expiry < datetime.now().date()
            record["expiry_date"] = expiry
        else:
            record["expiry_date"] = None

        # min_stock_level: Integer >= 0
        if i % 50 == 0:
            reorder_level = random.randint(-50, -1)
        elif i % 60 == 0:
            reorder_level = random.choice([99999, -999])
        else:
            if product_category == "high_turnover":
                reorder_level = random.randint(50, 200)
            elif product_category == "medium_turnover":
                reorder_level = random.randint(20, 100)
            elif product_category == "low_turnover":
                reorder_level = random.randint(5, 30)
            else:
                reorder_level = 0
        record["min_stock_level"] = reorder_level

        # current_stock: Integer >= 0, must be >= reserved_stock (with intentional violations)
        if i % 40 == 0:
            stock_qty = random.randint(-100, -1)
        elif i % 50 == 0:
            stock_qty = random.choice([99999, 1000000, -9999])
        elif i % 60 == 0:
            stock_qty = random.choice([10.5, 25.3, 100.75])
        elif i % 70 == 0:
            stock_qty = random.choice(["Unknown", "N/A", "Pending", None])
        else:
            if product_category == "high_turnover":
                stock_qty = random.choices(
                    [0, random.randint(1, 10), random.randint(11, 50), random.randint(51, 200), random.randint(201, 1000)],
                    weights=[5, 10, 30, 40, 15],
                    k=1,
                )[0]
            elif product_category == "medium_turnover": 
                stock_qty = random.choices(
                    [0, random.randint(1, 20), random.randint(21, 100), random.randint(101, 500)],
                    weights=[3, 20, 50, 27],
                    k=1,
                )[0]
            elif product_category == "low_turnover":
                stock_qty = random.choices(
                    [0, random.randint(1, 50), random.randint(51, 200), random.randint(201, 1000)],
                    weights=[2, 15, 40, 43],
                    k=1,
                )[0]
            else:
                stock_qty = random.choices(
                    [0, random.randint(1, 10), random.randint(11, 100)],
                    weights=[60, 30, 10],
                    k=1,
                )[0]
        record["current_stock"] = stock_qty

        if isinstance(stock_qty, (int, float)) and stock_qty > 0:
            max_reserved = min(int(stock_qty), stock_qty)
            reserved_qty = random.choices(
                [0, int(max_reserved * 0.1), int(max_reserved * 0.3), int(max_reserved * 0.5), int(max_reserved)],
                weights=[30, 25, 25, 15, 5],
                k=1,
            )[0]
        else:
            reserved_qty = 0
        record["reserved_stock"] = reserved_qty

        # available_qty: Must equal current_stock - reserved_stock exactly
        # If either is NULL/non-numeric, available_qty should be NULL
        if isinstance(stock_qty, (int, float)) and isinstance(reserved_qty, (int, float)):
            if isinstance(stock_qty, int) and isinstance(reserved_qty, int):
                available = stock_qty - reserved_qty
            else:
                available = stock_qty - reserved_qty
        else:
            available = None
        record["available_qty"] = available

        # last_restock_date: Must be >= created_date and <= today, YYYY-MM-DD (with violations)
        if i % 35 == 0:
            restocked_dt = fake.date_time_between(start_date="-6m", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            restocked = restocked_dt.strftime(random.choice(formats))
        elif i % 45 == 0:
            restocked = fake.date_time_between(start_date="+1d", end_date="+30d").strftime("%Y-%m-%d")
        elif i % 55 == 0:
            restocked = fake.date_time_between(start_date="-5y", end_date="-2y").strftime("%Y-%m-%d")
        else:
            if product_category == "high_turnover":
                restocked = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=7)), end_date="now").strftime("%Y-%m-%d")
            elif product_category == "medium_turnover": 
                restocked = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=30)), end_date="now").strftime("%Y-%m-%d")
            elif product_category == "low_turnover":
                restocked = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=90)), end_date="now").strftime("%Y-%m-%d")
            else:
                restocked = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=730)), end_date="now").strftime("%Y-%m-%d")
        record["last_restock_date"] = restocked

        # last_sale_date: Must be >= created_date and <= today, YYYY-MM-DD (with violations)
        if i % 50 == 0:
            last_sold = fake.date_time_between(start_date="+1d", end_date="+30d").strftime("%Y-%m-%d")
        else:
            if product_category == "high_turnover":
                last_sold = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=1)), end_date="now").strftime("%Y-%m-%d")
            elif product_category == "medium_turnover":
                last_sold = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=7)), end_date="now").strftime("%Y-%m-%d")
            elif product_category == "low_turnover":
                last_sold = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=30)), end_date="now").strftime("%Y-%m-%d")
            else:
                last_sold = fake.date_time_between(start_date=max(created_dt, datetime.now() - timedelta(days=365)), end_date="now").strftime("%Y-%m-%d")
        record["last_sale_date"] = last_sold

        # monthly_storage_cost: Decimal >= 0, scale 2, with non-numeric tokens
        if i % 45 == 0:
            storage_cost = round(random.uniform(-5, -0.1), 2)
        elif i % 55 == 0:
            storage_cost = random.choice([999.99, 0.001, -99])
        elif i % 65 == 0:
            storage_cost = random.choice(["Free", "Included", "N/A", None])
        else:
            storage_cost = round(random.uniform(0.10, 5.00), 2)
        record["monthly_storage_cost"] = storage_cost

        # warehouse_location: Mandatory, valid format, uppercase
        warehouse = random.choice(VALID_WAREHOUSES)
        record["warehouse_location"] = warehouse

        status = derive_stock_status(available, reorder_level, is_discontinued, is_expired)
        record["stock_status"] = status

        # total_stock_value: current_stock * unit_cost, must be 0.00 if stock is 0
        if isinstance(stock_qty, (int, float)) and stock_qty > 0:
            unit_cost = round(random.uniform(5, 200), 2)
            stock_value = round(stock_qty * unit_cost, 2)
            if i % 80 == 0:
                stock_value = round(stock_value * random.uniform(0.5, 1.5), 2)
        elif isinstance(stock_qty, (int, float)) and stock_qty == 0:
            stock_value = 0.00
        else:
            stock_value = None
        record["total_stock_value"] = stock_value

        # days_since_last_sale: Must equal (today - last_sale_date) in days
        try:
            last_sold_dt = datetime.strptime(last_sold, "%Y-%m-%d")
            days_since = (datetime.now() - last_sold_dt).days
            if i % 85 == 0:
                days_since = -random.randint(1, 30)
        except:
            days_since = None
        record["days_since_last_sale"] = days_since

        # restock_lead_time_days: Integer >= 1 (0 is violation)
        if i % 85 == 0:
            lead_time = random.randint(-10, -1)
        elif i % 95 == 0:
            lead_time = random.choice([0, 999, 365])
        else:
            lead_time = random.choices([1, 3, 7, 14, 30, 60, 90], weights=[10, 20, 30, 20, 10, 5, 5], k=1)[0]
        record["restock_lead_time_days"] = lead_time

        data.append(record)

    df = pd.DataFrame(data)

    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols: 
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[: 3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_inventory_data(number_of_inventories)
df = add_more_messiness(df)

output_file = "inventories.xlsx"
df.to_excel(output_file, index=False)

### Reviews Table Generator

In [24]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
MODERATION_STATUS_CANONICAL = ["Approved", "Pending", "Rejected", "Flagged"]
SUBMISSION_DEVICE_CANONICAL = [
    "Desktop",
    "Tablet",
    "iOS App",
    "Android App",
    "Mobile Web",
]
REVIEW_LANGUAGE_CANONICAL = ["en", "es", "fr", "de", "zh", "ja", "pt", "ru"]

# Rating step policy: integers only (1,2,3,4,5) with some half-stars as violations
VALID_RATINGS = [1, 2, 3, 4, 5]

# Spam/quality detection patterns
SPAM_TITLES = [
    "AMAZING!!!",
    "BEST EVER",
    "DO NOT BUY",
    "SCAM!!!",
    "Five Stars",
    "Good",
    "OK",
    "Nice",
    "👍",
    "⭐⭐⭐⭐⭐",
    "!!!!!!!!",
]
GENERIC_REVIEWS = [
    "Good product",
    "As expected",
    "Nice quality",
    "Fast shipping",
    "Would buy again",
    "Recommended",
    "Not bad",
    "Pretty good",
]
PLACEHOLDER_TEXT = ["undefined", "lorem ipsum", "test", "asdf", "N/A"]


def generate_messy_reviews_data(
    num_rows=3000, product_id_format="PROD", customer_id_format="CUST"
):
    data = []
    used_review_ids = []

    product_ids = [
        f"PROD_{i + starting_product_index}" for i in range(number_of_products)
    ]
    customer_ids = [
        f"CUST_{i + starting_customer_index}" for i in range(number_of_customers)
    ]

    customer_product_pairs = {}
    popular_products = product_ids[:20]

    for i in range(num_rows):
        record = {}

        # review_ref: Primary key, positive integer, unique
        if i % 53 == 0 and used_review_ids:
            review_id = random.choice(used_review_ids)  # Duplicate violation
        else:
            review_id = starting_review_index + i
            used_review_ids.append(review_id)
        record["review_ref"] = review_id

        # product_ref: Mandatory, FK to products
        if i % 59 == 0:
            prod_id = f"PROD_{9999}"  # FK violation
        else:
            if random.random() < 0.4:
                prod_id = random.choice(popular_products)
            else:
                prod_id = random.choice(product_ids)
        record["product_ref"] = prod_id

        # customer_ref: Mandatory (allow NULL for guest reviews), FK to customers
        is_guest = i % 17 == 0  # ~6% guest reviews
        if is_guest:
            cust_id = f"GUEST_{i}"  # Guest identifier instead of None
        elif i % 57 == 0:
            cust_id = f"CUST_{99999}"  # FK violation
        else:
            cust_id = random.choice(customer_ids)

        # Track duplicate reviews (same customer-product)
        if cust_id and prod_id:
            pair_key = f"{cust_id}_{prod_id}"
            if i % 31 == 0 and pair_key in customer_product_pairs:
                pass  # Duplicate review violation
            customer_product_pairs[pair_key] = True
        record["customer_ref"] = cust_id

        # star_rating: 1-5 inclusive, integers only (half-stars as violations)
        if i % 37 == 0:
            rating = random.choice(
                ["Five stars", "Good", "Bad", "****"]
            )  # String violation
        elif i % 48 == 0:
            rating = random.choice([0, 6, 10, -1, 100])  # Out of range violation
        elif i % 58 == 0:
            rating = random.choice([3.5, 4.5, 2.7, 1.8])  # Half-star violation
        else:
            # J-shaped distribution (more 5s and 1s)
            rating = random.choices([1, 2, 3, 4, 5], weights=[15, 5, 10, 25, 45], k=1)[
                0
            ]
        record["star_rating"] = rating

        # submitted_date: <= now(), no future dates
        if i % 36 == 0:
            review_date_dt = fake.date_time_between(start_date="-2y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y", "%d-%m-%Y"]
            review_date = review_date_dt.strftime(random.choice(formats))
        elif i % 46 == 0:
            review_date = fake.date_time_between(
                start_date="+1d", end_date="+30d"
            )  # Future violation
        elif i % 56 == 0:
            review_date = fake.date_time_between(
                start_date="-10y", end_date="-5y"
            )  # Very old
        else:
            days_ago = random.choices(
                [
                    random.randint(1, 7),
                    random.randint(8, 30),
                    random.randint(31, 180),
                    random.randint(181, 730),
                ],
                weights=[40, 30, 20, 10],
                k=1,
            )[0]
            review_date = datetime.now() - timedelta(days=days_ago)
        record["submitted_date"] = review_date

        # review_date_only: YYYY-MM-DD, must match submitted_date
        if isinstance(review_date, datetime):
            if i % 76 == 0:
                # Date mismatch violation
                record["review_date_only"] = (
                    review_date - timedelta(days=random.randint(1, 5))
                ).strftime("%Y-%m-%d")
            else:
                record["review_date_only"] = review_date.strftime("%Y-%m-%d")
        else:
            record["review_date_only"] = fake.date_between(
                start_date="-2y", end_date="today"
            ).strftime("%Y-%m-%d")

        # review_headline: Optional, 2-100 chars
        if i % 49 == 0:
            title = random.choice(SPAM_TITLES)  # Spam violation
        elif i % 69 == 0:
            title = fake.text(max_nb_chars=150)[:150]  # Too long violation (> 100)
        elif i % 79 == 0:
            title = "!!!!!!!!"  # Repeated punctuation violation
        elif i % 89 == 0:
            title = random.choice(
                ["undefined", "lorem ipsum", "error: stack trace"]
            )  # Template error
        else:
            # Generate realistic title based on rating
            if isinstance(rating, int):
                if rating >= 4:
                    title = random.choice(
                        [
                            "Great product!",
                            "Excellent quality",
                            "Highly recommend",
                            "Love it!",
                            "Perfect!",
                            "Exceeded expectations",
                            "Amazing value",
                            "Very satisfied",
                            fake.sentence(nb_words=4)[:80],
                        ]
                    )
                elif rating == 3:
                    title = random.choice(
                        [
                            "Decent product",
                            "It's okay",
                            "Average quality",
                            "Not bad",
                            "Could be better",
                            "Mixed feelings",
                            fake.sentence(nb_words=3)[:60],
                        ]
                    )
                else:
                    title = random.choice(
                        [
                            "Disappointed",
                            "Not worth it",
                            "Poor quality",
                            "Waste of money",
                            "Do not recommend",
                            "Terrible experience",
                            fake.sentence(nb_words=3)[:60],
                        ]
                    )
            else:
                title = fake.sentence(nb_words=4)[:80]
        record["review_headline"] = title

        # review_content: Minimum 10-20 chars, quality checks
        if i % 42 == 0:
            text = random.choice(GENERIC_REVIEWS)  # Generic/low quality
        elif i % 52 == 0:
            # Spam with URL/email
            text = f"Check out {fake.url()} for deals! Contact {fake.email()} for info."
        elif i % 62 == 0:
            # Repetitive text violation
            word = random.choice(["GREAT", "BAD", "LOVE", "HATE"])
            text = f"{word} " * random.randint(10, 50)
        elif i % 72 == 0:
            text = fake.text(max_nb_chars=5000)  # Very long
        elif i % 82 == 0:
            # Non-English (language mismatch potential)
            text = random.choice(
                [
                    "很好的产品！强烈推荐。",
                    "Très bon produit, je recommande!",
                    "отличный продукт",
                    "素晴らしい製品です",
                ]
            )
        elif i % 92 == 0:
            # Content doesn't match rating
            if isinstance(rating, int) and rating >= 4:
                text = "Terrible product.Very disappointed.Would not buy again.Waste of money."
            elif isinstance(rating, int) and rating <= 2:
                text = "This is the best product I've ever purchased! Absolutely love it! Perfect!"
            else:
                text = fake.paragraph(nb_sentences=3)
        else:
            # Valid: realistic review based on rating
            if isinstance(rating, int):
                if rating >= 4:
                    text = fake.paragraph(nb_sentences=random.randint(2, 5))
                    text += random.choice(
                        [" Highly recommend!", " Would buy again.", " Great value.", ""]
                    )
                elif rating == 3:
                    text = fake.paragraph(nb_sentences=random.randint(2, 4))
                    text += random.choice(
                        [" It's okay for the price.", " Has pros and cons.", ""]
                    )
                else:
                    text = fake.paragraph(nb_sentences=random.randint(1, 3))
                    text += random.choice(
                        [" Very disappointed.", " Not worth the money.", ""]
                    )
            else:
                text = fake.paragraph(nb_sentences=random.randint(2, 4))
        record["review_content"] = text

        # verified_purchase: Boolean, hard rule: TRUE only if customer_ref non-null
        if i % 38 == 0:
            verified = random.choice(["Y", "N", "Yes", "No", "1", "0", "true", "false"])
        elif i % 60 == 0:
            # Guest but verified violation
            if is_guest:
                verified = True
            else:
                verified = False
        else:
            # Valid: verified only if customer exists
            if not is_guest:
                verified = random.choices([True, False], weights=[70, 30], k=1)[0]
            else:
                verified = False
        record["verified_purchase"] = verified

        # helpful_count: Integer >= 0
        if i % 64 == 0:
            helpful = random.randint(-10, -1)  # Negative violation
        elif i % 74 == 0:
            # Suspicious: too many votes for new review
            if (
                isinstance(review_date, datetime)
                and (datetime.now() - review_date).days < 7
            ):
                helpful = random.randint(100, 1000)
            else:
                helpful = random.randint(0, 50)
        else:
            # Valid: older reviews have more votes
            if isinstance(review_date, datetime):
                days_old = (datetime.now() - review_date).days
                max_votes = min(days_old // 10, 100)
                helpful = random.randint(0, max(max_votes, 1))
            else:
                helpful = random.randint(0, 20)
        record["helpful_count"] = helpful

        # total_votes: Integer >= 0, >= helpful_count
        if isinstance(helpful, int) and helpful >= 0:
            if i % 84 == 0:
                # total < helpful violation
                total = max(0, helpful - random.randint(1, 5))
            else:
                # Valid: total >= helpful
                total = helpful + random.randint(0, 20)
        else:
            total = random.randint(0, 30)
        record["total_votes"] = total

        # image_count: Integer >= 0, typically 0-10
        if i % 81 == 0:
            images = random.randint(-5, -1)  # Negative violation
        elif i % 91 == 0:
            images = random.randint(15, 50)  # > 10 suspicious
        else:
            # Valid: 0-10
            images = random.choices(
                [0, 1, 2, 3, 4, 5], weights=[40, 30, 15, 10, 3, 2], k=1
            )[0]
        record["image_count"] = images

        # reviewer_name: 1-80 chars, defaulting rules
        if i % 65 == 0:
            name = random.choice(
                ["123456", "@#$%"]
            )  # Invalid (only symbols/digits)
        elif i % 75 == 0:
            name = fake.email()  # Email pattern violation
        elif i % 85 == 0:
            name = "A" * 100  # Too long violation
        else:
            # Valid with defaulting
            if verified == True:
                name = "Verified Reviewer"
            elif is_guest:
                name = random.choice(
                    ["A Customer", "Anonymous Shopper", fake.first_name()]
                )
            else:
                name = fake.name()
        record["reviewer_name"] = name

        # moderation_status: Canonical set
        if i % 73 == 0:
            status = random.choice(["Hidden", "Needs Review", "Live"])  # Non-canonical
        else:
            # Valid: mostly Approved
            status = random.choices(
                MODERATION_STATUS_CANONICAL, weights=[80, 10, 5, 5], k=1
            )[0]
        record["moderation_status"] = status

        # seller_response: Optional, 5-2000 chars
        if random.random() > 0.9: # 10% have response
            if i % 90 == 0:
                # Response on non-approved review violation
                if status in ["Pending", "Rejected"]:
                    response = "Thank you for your feedback!"
                else:
                    response = "We appreciate your review!"
            else:
                # Valid response
                if isinstance(rating, int) and rating <= 2:
                    response = f"We're sorry about your experience.Please contact support@{fake.domain_name()} for help."
                else:
                    response = "Thank you for your review! We appreciate your feedback."
            record["seller_response"] = response

        # review_language: ISO 639-1 two-letter
        if i % 78 == 0:
            lang = random.choice(["English", "Spanish", "French"])  # Invalid format
        elif i % 88 == 0:
            # Language mismatch with content
            if text and ("很好" in str(text) or "Très" in str(text)):
                lang = "en"  # Mismatch violation
            else:
                lang = random.choice(["zh", "fr", "ja"])
        else:
            # Valid: mostly English
            lang = random.choices(
                REVIEW_LANGUAGE_CANONICAL, weights=[70, 10, 5, 5, 3, 3, 2, 2], k=1
            )[0]
        record["review_language"] = lang

        # submission_device: Canonical set
        if i % 87 == 0:
            device = random.choice(["iPhone", "Android", "Mobile"])  # Non-canonical
        else:
            device = random.choice(SUBMISSION_DEVICE_CANONICAL)
        record["submission_device"] = device

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_reviews_data(number_of_reviews)
df = add_more_messiness(df)

output_file = "reviews.xlsx"
df.to_excel(output_file, index=False)

### Marketing Campaigns Generator

In [25]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
import random

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
CHANNEL_TYPES_CANONICAL = [
    "Email",
    "Social Media",
    "Search",
    "Display",
    "Video",
    "SMS",
    "Radio",
    "TV",
    "Direct Mail",
]
DIGITAL_CHANNELS = ["Email", "Social Media", "Search", "Display", "Video", "SMS"]
OFFLINE_CHANNELS = ["Radio", "TV", "Direct Mail"]
STATUS_CANONICAL = ["Active", "Paused", "Completed", "Planned", "Cancelled"]
PLATFORM_CANONICAL = ["Google Ads", "Meta Ads", "TikTok", "LinkedIn", "Instagram"]
VARIANT_CANONICAL = ["Control", "Variant A", "Variant B", "Variant C"]

CAMPAIGN_THEMES = [
    "Summer Sale",
    "Black Friday",
    "Christmas Special",
    "New Year Deal",
    "Spring Collection",
    "Back to School",
    "Flash Sale",
    "Clearance",
    "Product Launch",
    "Brand Awareness",
    "Customer Retention",
    "Lead Generation",
    "Holiday Special",
    "Anniversary Sale",
]

AUDIENCE_SEGMENTS = [
    "High_LTV",
    "Cart_Abandoners",
    "New_Customers_30d",
    "Women_25_40_USA",
    "Men_18_35_Urban",
    "Parents_with_children",
    "High_income_households",
    "College_students",
    "Senior_65_plus",
    "Millennials_Tech_savvy",
    "Gen_Z_Social",
    "B2B_Decision_makers",
    "Small_business_owners",
    "Fitness_enthusiasts",
    "Premium_customers",
    "First_time_buyers",
    "Loyal_customers",
]


def derive_status_from_dates(launch_date, completion_date):
    """Derive campaign status from dates per rules."""
    today = date.today()
    if not isinstance(launch_date, date) or not isinstance(completion_date, date):
        return random.choice(["Active", "Paused", "Completed"])
    if launch_date > today:
        return "Planned"
    if completion_date < today:
        return "Completed"
    return random.choice(["Active", "Paused"])


def generate_messy_marketing_campaigns_data(num_rows=500):
    data = []
    used_campaign_ids = []

    for i in range(num_rows):
        record = {}

        # campaign_ref: Primary key, positive integer, unique
        if i % 50 == 0 and used_campaign_ids:
            campaign_id = random.choice(used_campaign_ids)  # Duplicate violation
        else:
            campaign_id = starting_campaign_index + i
            used_campaign_ids.append(campaign_id)
        record["campaign_ref"] = campaign_id

        # campaign_title: 5-100 chars, non-null, non-placeholder
        if i % 45 == 0:
            name = fake.text(max_nb_chars=150)[:120]  # Exceeds 100
        elif i % 55 == 0:
            name = random.choice(["Campaign #1", "Sale!!!", "50% OFF", "MEGA SALE"])
        elif i % 65 == 0:
            name = "Summer Sale 2025"  # Duplicate name
        else:
            theme = random.choice(CAMPAIGN_THEMES)
            year = random.choice(["2024", "2025"])
            suffix = random.choice(["", " - Phase 1", " - Final"])
            name = f"{theme} {year}{suffix}"
            # Ensure 5-100 chars
            if len(name) < 5:
                name = name + " Campaign"
        record["campaign_title"] = name

        # channel_type: Canonical set only
        if i % 30 == 0:
            camp_type = random.choice(
                ["email", "EMAIL", "Email Marketing", "E-mail"]
            )  # Case violation
        elif i % 50 == 0:
            camp_type = random.choice(
                ["Emal", "Socail Media", "PPG", "Displya"]
            )  # Typos
        else:
            camp_type = random.choice(CHANNEL_TYPES_CANONICAL)
        record["channel_type"] = camp_type

        is_digital = camp_type in DIGITAL_CHANNELS
        is_offline = camp_type in OFFLINE_CHANNELS

        # launch_date: YYYY-MM-DD format
        if i % 40 == 0:
            start_dt = fake.date_between(start_date="-1y", end_date="+6m")
            formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            start_date = start_dt.strftime(random.choice(formats))
        elif i % 60 == 0:
            start_date = fake.date_between(
                start_date="-10y", end_date="-5y"
            )  # Very old
        elif i % 70 == 0:
            start_date = fake.date_between(
                start_date="+2y", end_date="+5y"
            )  # Far future
        else:
            start_date = fake.date_between(start_date="-6m", end_date="+3m")
        record["launch_date"] = start_date

        # completion_date: Must be > launch_date (with violations)
        if i % 35 == 0: # Violation: end before start
            if isinstance(start_date, date):
                end_date = start_date - timedelta(days=random.randint(1, 30))
            else:
                end_date = fake.date_between(start_date="-2y", end_date="-1y")
        elif i % 45 == 0:
            end_date = start_date  # Same day - violation of strictly greater
        elif i % 55 == 0:
            end_date = fake.date_between(
                start_date="+10y", end_date="+20y"
            )  # >365 days
        elif i % 65 == 0:
            end_dt = fake.date_between(start_date="-3m", end_date="+6m")
            end_date = end_dt.strftime("%m/%d/%Y")  # String format
        else:
            # Valid: completion_date > launch_date, duration 1-365 days
            if isinstance(start_date, date):
                duration = random.choices(
                    [
                        random.randint(1, 7),
                        random.randint(8, 30),
                        random.randint(31, 90),
                        random.randint(91, 365),
                    ],
                    weights=[20, 40, 30, 10],
                    k=1,
                )[0]
                end_date = start_date + timedelta(days=duration)
            else:
                end_date = fake.date_between(start_date="-2m", end_date="+6m")
        record["completion_date"] = end_date

        # campaign_status: Canonical set, date-consistent
        if i % 30 == 0:
            status = random.choice(
                ["active", "ACTIVE", "Running", "Live", "1"]
            )  # Case violation
        elif i % 40 == 0:
            status = random.choice(
                ["Pending", "Draft", "Archived", "Deleted"]
            )  # Invalid
        elif i % 50 == 0:
            status = random.choice(["Activ", "Pasued", "Complted"])  # Typos
        else:
            # Valid: derive from dates
            status = derive_status_from_dates(start_date, end_date)
        record["campaign_status"] = status

        # allocated_budget: Decimal >= 0
        if i % 37 == 0:
            budget = round(random.uniform(-10000, -100), 2)  # Negative violation
        elif i % 47 == 0:
            budget = 0  # Zero budget
        elif i % 53 == 0:
            budget = random.choice([999999999.99, 0.01, -99999])  # Extreme
        else:
            # Valid: realistic budget based on channel
            if camp_type in ["TV", "Radio"]:
                budget = round(random.uniform(50000, 500000), 2)
            elif camp_type in ["Search", "Display"]:
                budget = round(random.uniform(1000, 50000), 2)
            elif camp_type in ["Email", "SMS"]:
                budget = round(random.uniform(100, 10000), 2)
            else:
                budget = round(random.uniform(500, 25000), 2)
        record["allocated_budget"] = budget

        # current_spend: Decimal >= 0, typically <= budget * 1.10
        if i % 41 == 0:
            spent = round(random.uniform(-5000, -10), 2)  # Negative violation
        elif i % 51 == 0: # Overspend > 110%
            if isinstance(budget, (int, float)) and budget > 0:
                spent = round(budget * random.uniform(1.15, 2.0), 2)
            else:
                spent = round(random.uniform(10000, 50000), 2)
        elif i % 61 == 0:
            spent = 0  # Zero spent
        else:
            # Valid: spend based on status and budget
            if isinstance(budget, (int, float)) and budget > 0:
                if status == "Planned":
                    spent = 0  # Not started
                elif status == "Completed":
                    spent = round(budget * random.uniform(0.7, 1.0), 2)
                elif status == "Cancelled":
                    spent = round(budget * random.uniform(0, 0.1), 2)  # Minimal
                else: # Active/Paused
                    spent = round(budget * random.uniform(0.3, 0.9), 2)
            else:
                spent = round(random.uniform(100, 10000), 2)
        record["current_spend"] = spent

        # total_impressions: Integer >= 0 (digital channels should have this)
        if i % 49 == 0:
            impressions = random.randint(-10000, -1)  # Negative violation
        elif i % 59 == 0:
            impressions = 0  # Zero but will have clicks - violation setup
        elif i % 67 == 0:
            impressions = random.choice([999999999, 0.5, -99999])  # Extreme/decimal
        else:
            # Valid: impressions based on spend and channel
            if is_offline:
                impressions = random.randint(10000, 1000000)
            elif isinstance(spent, (int, float)) and spent > 0:
                if camp_type in ["Email", "SMS"]:
                    impressions = int(spent * random.uniform(10, 50))
                elif camp_type in ["Display", "Social Media"]:
                    impressions = int(spent * random.uniform(100, 500))
                else:
                    impressions = int(spent * random.uniform(50, 200))
            else:
                impressions = random.randint(1000, 100000)
        record["total_impressions"] = impressions

        # total_clicks: Integer >= 0, must be <= impressions
        if i % 45 == 0:
            clicks = random.randint(-1000, -1)  # Negative violation
        elif i % 55 == 0: # clicks > impressions violation
            if isinstance(impressions, int) and impressions > 0:
                clicks = int(impressions * random.uniform(1.1, 2.0))
            else:
                clicks = random.randint(10000, 50000)
        elif i % 65 == 0:
            clicks = random.choice([100.5, 250.75, 1000.25])  # Decimal
        else:
            # Valid: clicks <= impressions, realistic CTR 0.5%-5%
            if isinstance(impressions, int) and impressions > 0:
                ctr = random.uniform(0.005, 0.05)
                clicks = int(impressions * ctr)
            else:
                clicks = random.randint(10, 5000)
        record["total_clicks"] = clicks

        # conversion_count: Integer >= 0, must be <= clicks
        if i % 50 == 0:
            conversions = random.randint(-100, -1)  # Negative violation
        elif i % 60 == 0: # conversions > clicks violation
            if isinstance(clicks, (int, float)) and clicks > 0:
                conversions = int(clicks * random.uniform(1.1, 2.0))
            else:
                conversions = random.randint(1000, 5000)
        elif i % 70 == 0:
            conversions = clicks  # 100% conversion - suspicious
        else:
            # Valid: conversions <= clicks, realistic 1%-10%
            if isinstance(clicks, (int, float)) and clicks > 0:
                conv_rate = random.uniform(0.01, 0.10)
                conversions = int(clicks * conv_rate)
            else:
                conversions = random.randint(0, 500)
        record["conversion_count"] = conversions

        # ctr_rate: (clicks / impressions) * 100, range 0-100
        if i % 80 == 0:
            ctr = random.uniform(101, 200)  # Invalid > 100%
        elif (
            all(isinstance(x, (int, float)) for x in [clicks, impressions])
            and impressions > 0
        ):
            ctr = round((clicks / impressions) * 100, 2)
        elif isinstance(impressions, (int, float)) and impressions == 0:
            ctr = 0  # Division by zero rule
        else:
            ctr = 0
        record["ctr_rate"] = ctr

        # conversion_rate: (conversions / clicks) * 100, range 0-100
        if i % 85 == 0:
            conv_rate = random.uniform(101, 150)  # Invalid > 100%
        elif (
            all(isinstance(x, (int, float)) for x in [conversions, clicks])
            and clicks > 0
        ):
            conv_rate = round((conversions / clicks) * 100, 2)
        elif isinstance(clicks, (int, float)) and clicks == 0:
            conv_rate = 0  # Division by zero rule
        else:
            conv_rate = 0
        record["conversion_rate"] = conv_rate

        # avg_cpc: current_spend / total_clicks
        if i % 90 == 0:
            cpc = random.choice([0, 1000, -10])  # Extreme/negative
        elif (
            all(isinstance(x, (int, float)) for x in [spent, clicks]) and clicks > 0
        ):
            cpc = round(spent / clicks, 2)
        elif isinstance(clicks, (int, float)) and clicks == 0:
            if isinstance(spent, (int, float)) and spent == 0:
                cpc = 0  # Both zero
            else:
                cpc = 0  # Spend but no clicks
        else:
            cpc = 0
        record["avg_cpc"] = cpc

        # roi_percentage: ((revenue - spend) / spend) * 100, requires spend > 0
        if i % 75 == 0:
            roi = random.choice([-100, 10000, 99999])  # Extreme
        elif status in ["Planned"]:
            roi = 0  # Not applicable for planned
        elif (
            all(isinstance(x, (int, float)) for x in [spent, conversions])
            and spent > 0
        ):
            avg_order_value = random.uniform(50, 200)
            revenue = conversions * avg_order_value
            roi = round(((revenue - spent) / spent) * 100, 2)
        elif isinstance(spent, (int, float)) and spent == 0:
            roi = 0  # Division undefined
        else:
            roi = 0
        record["roi_percentage"] = roi

        # target_segment: Non-empty for Active campaigns
        if i % 45 == 0:
            audience = fake.text(max_nb_chars=1000)  # Very long
        else:
            # Valid: standardized segment codes
            segments = random.sample(AUDIENCE_SEGMENTS, random.randint(1, 3))
            audience = ", ".join(segments)
        record["target_segment"] = audience

        # campaign_manager: Non-null for Active/Completed
        owner = fake.name()
        record["campaign_manager"] = owner

        # ad_platform: Canonical set, must align with channel_type
        if is_digital and camp_type in ["Social Media", "Search", "Display", "Video"]:
            # Valid: platform aligned with channel
            if camp_type == "Search":
                platform = "Google Ads"
            elif camp_type == "Social Media":
                platform = random.choice(["Meta Ads", "LinkedIn", "TikTok"])
            elif camp_type == "Video":
                platform = random.choice(["Google Ads", "Meta Ads", "TikTok"])
            else:
                platform = random.choice(PLATFORM_CANONICAL)
            record["ad_platform"] = platform
        elif is_offline:
            # Offline channels - use a placeholder platform
            record["ad_platform"] = "Offline"
        else:
            record["ad_platform"] = random.choice(PLATFORM_CANONICAL)

        # test_variant: Consistent naming
        if i % 80 == 0:
            variant = random.choice(["Test", "Winner", "Original"])  # Non-canonical
        else:
            variant = random.choice(VARIANT_CANONICAL)
        record["test_variant"] = variant

        # created_timestamp: <= now(), <= launch_date
        if i % 40 == 0:
            created = fake.date_time_between(
                start_date="-1y", end_date="now"
            ).strftime("%Y-%m-%d %H:%M:%S")
        elif i % 50 == 0: # Violation: created after launch
            if isinstance(start_date, date):
                created = fake.date_time_between(
                    start_date=start_date + timedelta(days=1),
                    end_date=start_date + timedelta(days=30),
                )
            else:
                created = fake.date_time_between(start_date="+1d", end_date="+30d")
        else:
            # Valid: created before launch_date
            if isinstance(start_date, date):
                created = fake.date_time_between(
                    start_date=start_date - timedelta(days=60),
                    end_date=start_date - timedelta(days=1),
                )
            else:
                created = fake.date_time_between(start_date="-1y", end_date="now")
        record["created_timestamp"] = created

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_marketing_campaigns_data(number_of_campaigns)
df = add_more_messiness(df)

output_file = "marketing_campaigns.xlsx"
df.to_excel(output_file, index=False)

### Customer Sessions Table Generator

In [26]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, timezone
import random
import string
import uuid
import re

fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# System launch date for validation
SYSTEM_LAUNCH_DATE = datetime(2020, 1, 1, tzinfo=timezone.utc)

# Controlled vocabularies
CANONICAL_DEVICES = {"Mobile", "Desktop", "Tablet", "Smart TV", "Wearable"}
CANONICAL_TRAFFIC_SOURCES = {
    "Organic Search",
    "Paid Search",
    "Social Media",
    "Direct",
    "Email",
    "Referral",
    "Display Ads",
}
CANONICAL_BROWSERS = {"Chrome", "Safari", "Edge", "Firefox", "Opera", "Samsung Browser"}
CANONICAL_COUNTRIES = {
    "US",
    "GB",
    "CA",
    "AU",
    "DE",
    "FR",
    "JP",
    "IN",
    "BR",
    "MX",
    "IT",
    "ES",
    "NL",
    "SE",
    "CH",
    "NZ",
    "SG",
    "HK",
    "KR",
    "CN",
}


def generate_session_ref():
    """Generate unique UUID v4 format session reference."""
    return str(uuid.uuid4()).lower()


def to_iso8601(dt):
    """Convert to ISO-8601 format with UTC timezone."""
    if dt is None:
        return None
    if isinstance(dt, str):
        try:
            parsed = pd.to_datetime(dt, errors="coerce")
            if pd.isna(parsed):
                return None
            return parsed.strftime("%Y-%m-%dT%H:%M:%SZ")
        except:
            return None
    if isinstance(dt, (datetime, pd.Timestamp)):
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return None


def normalize_bool(value):
    """Normalize to boolean: Y/N, yes/no, true/false, 1/0."""
    if value is None:
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        value_lower = value.lower().strip()
        if value_lower in ["y", "yes", "true", "t", "1"]:
            return True
        elif value_lower in ["n", "no", "false", "f", "0"]:
            return False
    if isinstance(value, (int, float)):
        return bool(value)
    return False


def calculate_session_duration(start_ts, end_ts):
    """Calculate session duration in seconds."""
    if start_ts is None or end_ts is None:
        return 0
    try:
        start = pd.to_datetime(start_ts)
        end = pd.to_datetime(end_ts)
        if end < start:
            return 0
        return int((end - start).total_seconds())
    except:
        return 0


def generate_messy_customer_sessions_data(num_rows=5000, customer_id_format="CUST"):
    """Generate customer sessions with all 14 validation rules integrated."""
    data = []
    used_session_refs = set()
    valid_customer_ids = set()
    customer_sessions = {}  # Track sessions per customer for visitor_type

    # Generate pool of valid customer IDs
    for i in range(number_of_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{starting_customer_index + i}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{starting_customer_index + i}"
        else:
            cust_id = str(starting_customer_index + i)
        valid_customer_ids.add(cust_id)

    for i in range(num_rows):
        record = {}
        idx = starting_session_index + i

        # 1.session_ref: UUID v4, unique, no duplicates
        session_ref = generate_session_ref()
        while session_ref in used_session_refs:
            session_ref = generate_session_ref()
        record["session_ref"] = session_ref
        used_session_refs.add(session_ref)

        # 2.user_id: Guest gets GUEST_ prefix, referential integrity for logged-in
        is_guest = random.random() < 0.35  # 35% guest sessions

        if is_guest:
            user_id = f"GUEST_{i}"  # Guest identifier instead of None
        elif i % 60 == 0:
            # Invalid/non-existent customer (violation for testing)
            if customer_id_format == "CUST":
                user_id = f"CUST_99999"
            elif customer_id_format == "CUSTOMER":
                user_id = f"CUSTOMER-99999"
            else:
                user_id = "99999"
        else:
            user_id = random.choice(list(valid_customer_ids))

        # Track sessions per customer
        if user_id and not user_id.startswith("GUEST_"):
            if user_id not in customer_sessions:
                customer_sessions[user_id] = []
            customer_sessions[user_id].append(i)

        record["user_id"] = user_id

        # 3.start_timestamp: ISO-8601, non-null, >= system launch
        if i % 35 == 0:
            # Future date (violation for testing)
            start_ts = fake.date_time_between(start_date="+1d", end_date="+7d")
        elif i % 45 == 0:
            # Too old (violation)
            start_ts = fake.date_time_between(start_date="-3y", end_date="-2y")
        else:
            start_ts = fake.date_time_between(
                start_date=SYSTEM_LAUNCH_DATE, end_date="now"
            )

        start_iso = to_iso8601(start_ts)
        record["start_timestamp"] = start_iso

        # 3.end_timestamp: ISO-8601, >= start_timestamp, <= 4 hours
        if i % 40 == 0:
            # End before start (violation)
            if isinstance(start_ts, datetime):
                end_ts = start_ts - timedelta(minutes=random.randint(1, 60))
            else:
                end_ts = start_ts
        elif i % 50 == 0:
            # Very long session > 4 hours (violation)
            if isinstance(start_ts, datetime):
                end_ts = start_ts + timedelta(hours=random.randint(5, 24))
            else:
                end_ts = start_ts
        else:
            if isinstance(start_ts, datetime):
                # Realistic duration: 1-240 minutes (4 hours)
                duration_minutes = random.choices(
                    [
                        random.randint(1, 5),  # Bounce
                        random.randint(6, 15),  # Quick
                        random.randint(16, 30),  # Normal
                        random.randint(31, 60),  # Engaged
                        random.randint(61, 240),  # Very engaged
                    ],
                    weights=[35, 25, 20, 15, 5],
                    k=1,
                )[0]
                end_ts = start_ts + timedelta(minutes=duration_minutes)
            else:
                end_ts = start_ts

        # Convert to ISO-8601
        end_iso = to_iso8601(end_ts)
        record["end_timestamp"] = end_iso

        # 11.session_duration_sec: Must equal (end - start) in seconds
        duration_sec = calculate_session_duration(start_iso, end_iso)
        record["session_duration_sec"] = duration_sec

        # 4.device_category: Canonical values
        if i % 35 == 0:
            device = random.choice(["mobile", "MOBILE", "deskop", "bot"])  # Typos/case violations
        else:
            device = random.choices(
                ["Mobile", "Desktop", "Tablet", "Smart TV", "Wearable"],
                weights=[45, 40, 10, 3, 2],
                k=1,
            )[0]
        record["device_category"] = device

        # 5.traffic_source: Canonical values
        if i % 30 == 0:
            source = random.choice(["organic search", "PAID SEARCH", "socail media"])  # Case/typo violations
        else:
            source = random.choice(list(CANONICAL_TRAFFIC_SOURCES))
        record["traffic_source"] = source

        # 6.page_views: Integer >= 1
        if i % 45 == 0:
            page_views = 0  # Must be >= 1 violation
        else:
            if duration_sec and duration_sec > 0:
                # Realistic: 1-50 pages based on duration
                if duration_sec < 300: # < 5 min
                    page_views = random.randint(1, 2)
                elif duration_sec < 900: # < 15 min
                    page_views = random.randint(2, 5)
                elif duration_sec < 1800: # < 30 min
                    page_views = random.randint(4, 10)
                else:
                    page_views = random.randint(8, 30)
            else:
                page_views = random.randint(1, 15)

        # Ensure >= 1 for most cases
        if isinstance(page_views, int) and page_views < 1 and i % 45 != 0:
            page_views = 1

        record["page_views"] = page_views

        # 6.products_browsed: Integer >= 0, <= page_views
        if i % 40 == 0:
            # products > page_views violation
            if isinstance(page_views, int) and page_views > 0:
                products = page_views + random.randint(1, 5)
            else:
                products = random.randint(5, 10)
        else:
            if isinstance(page_views, int) and page_views > 0:
                # products <= pages
                max_products = max(1, int(page_views * 0.6))
                products = random.randint(0, max_products)
            else:
                products = random.randint(0, 5)

        record["products_browsed"] = products

        # 7.purchase_made: Boolean
        if i % 45 == 0:
            purchase = random.choice(["maybe", "pending"])  # String violations
        else:
            # Realistic: more likely if products viewed
            if isinstance(products, int) and products >= 5:
                purchase = random.choices([True, False], weights=[30, 70], k=1)[0]
            elif isinstance(products, int) and products >= 2:
                purchase = random.choices([True, False], weights=[15, 85], k=1)[0]
            else:
                purchase = random.choices([True, False], weights=[2, 98], k=1)[0]

        purchase_bool = normalize_bool(purchase) if not isinstance(purchase, str) else False
        record["purchase_made"] = purchase

        # 7.cart_abandoned: Boolean, mutual exclusivity with purchase
        if i % 80 == 0:
            # Violation: both purchase and abandoned true
            abandoned = True
        else:
            # Can't abandon if converted
            if purchase_bool is True:
                abandoned = False
            else:
                # Higher chance if products viewed
                if isinstance(products, int) and products > 0:
                    abandoned = random.choices([True, False], weights=[40, 60], k=1)[0]
                else:
                    abandoned = False

        record["cart_abandoned"] = abandoned

        # 7.bounce_session: Boolean, page_views = 1
        if isinstance(page_views, int):
            bounce = page_views == 1
            # Consistency: if bounce, page_views must = 1
            if bounce and page_views != 1:
                bounce = False
        else:
            bounce = False

        record["bounce_session"] = bounce

        # 8.geo_country: ISO Alpha-2
        if i % 35 == 0:
            country = random.choice(["UK", "USA", "United States"])  # Invalid format violations
        else:
            country = random.choice(list(CANONICAL_COUNTRIES))
        record["geo_country"] = country

        # 9.entry_page: Valid path
        if i % 40 == 0:
            entry = random.choice(["home", "products", "www.example.com/page"])  # Missing / prefix violations
        else:
            entry = random.choice(
                [
                    "/home",
                    "/products",
                    "/sale",
                    "/search",
                    "/category/electronics",
                    "/product/item-123",
                    "/cart",
                    "/checkout",
                ]
            )
        record["entry_page"] = entry

        # 9.exit_page: Valid path, consistency checks
        if i % 45 == 0:
            exit_page = random.choice(["checkout", "cart", "www.example.com/exit"])  # Missing / prefix violations
        else:
            # Consistency: if page_views = 1, entry_page = exit_page
            if isinstance(page_views, int) and page_views == 1 and entry:
                exit_page = entry
            elif purchase_bool is True:
                exit_page = "/order-confirmation"
            elif abandoned is True:
                exit_page = random.choice(["/cart", "/checkout"])
            else:
                exit_page = random.choice(
                    ["/home", "/products", "/product/item-456", "/about", "/contact"]
                )
        record["exit_page"] = exit_page

        # 10.visitor_type: New or Returning
        if user_id and not user_id.startswith("GUEST_"):
            # Check if returning
            if user_id in customer_sessions and len(customer_sessions[user_id]) > 1:
                visitor = "Returning"
            else:
                visitor = "New"
        else:
            # Guest sessions can be New or Returning
            visitor = random.choice(["New", "Returning"])

        if i % 90 == 0:
            visitor = random.choice(["new", "RETURNING", "returning"])  # Case violations

        record["visitor_type"] = visitor

        # 12.cart_items_count: Integer >= 0
        if purchase_bool is True or abandoned is True:
            if i % 80 == 0:
                cart_count = random.randint(-5, -1)  # Negative violation
            else:
                cart_count = random.randint(1, 10)
        else:
            cart_count = 0

        record["cart_items_count"] = cart_count

        # 13.browser_name: Canonical values
        if i % 40 == 0:
            browser = random.choice(["chrome mobile", "CHROME", "Crhome", "safari ios"])  # Case/typo violations
        else:
            browser = random.choice(list(CANONICAL_BROWSERS))
        record["browser_name"] = browser

        # 14.session_revenue: Decimal 2 decimals, >= 0, dependency on purchase_made
        if purchase_bool is True:
            if i % 95 == 0:
                revenue = round(random.uniform(-100, 0), 2)  # Negative violation
            else:
                revenue = round(random.uniform(20, 1000), 2)
        else:
            if i % 85 == 0:
                # Violation: revenue without purchase
                revenue = round(random.uniform(10, 500), 2)
            else:
                revenue = 0.00

        # Ensure 2 decimal places
        if isinstance(revenue, (int, float)):
            revenue = round(revenue, 2)

        record["session_revenue"] = revenue

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_more_messiness(df):
    """Add additional data quality issues."""
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


if __name__ == "__main__":
    df = generate_messy_customer_sessions_data(number_of_sessions, "CUST")
    df = add_more_messiness(df)

    output_file = "customer_sessions.xlsx"
    df.to_excel(output_file, index=False)

### Supplier Table Generator

In [27]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
import random

fake = Faker(["en_US", "en_GB", "de_DE", "fr_FR"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Canonical values
STATUS_CANONICAL = ["Active", "Inactive", "Suspended", "Pending", "Under Review"]
COUNTRY_CODES = [
    "US",
    "GB",
    "DE",
    "FR",
    "CA",
    "AU",
    "JP",
    "CN",
    "IN",
    "BR",
    "MX",
    "IT",
    "ES",
    "NL",
]

# Rating text to numeric mapping
RATING_TEXT_MAP = {
    "Excellent": 5.00,
    "Good": 4.00,
    "Average": 3.00,
    "Fair": 3.00,
    "Poor": 2.00,
    "Bad": 1.00,
}


def generate_messy_supplier_data(num_rows=1000):
    data = []
    used_ids = []
    used_tax_ids = []
    duplicate_names = []

    for i in range(num_rows):
        record = {}

        # supplier_id: Primary key, format ^SUP_[1-9][0-9]*$, positive, unique
        if i % 53 == 0 and used_ids:
            supplier_id = random.choice(used_ids)  # Duplicate violation
        elif i % 61 == 0:
            # Format violations
            formats = [
                f"S{i + starting_supplier_index}",
                f"VND-{i + starting_supplier_index}",
                f"PARTNER_{i + starting_supplier_index}",
                str(i + starting_supplier_index),
            ]
            supplier_id = random.choice(formats)
        elif i % 73 == 0:
            # Case inconsistencies
            supplier_id = random.choice(
                [
                    f"sup_{i + starting_supplier_index}",
                    f"Sup_{i + starting_supplier_index}",
                ]
            )
        elif i % 83 == 0:
            # Placeholder values
            supplier_id = random.choice(["SUP_0", "SUP_999999", "SUP_TEST"])
        else:
            # Valid: canonical format
            supplier_id = f"SUP_{i + starting_supplier_index}"

        used_ids.append(supplier_id)
        record["supplier_id"] = supplier_id

        # supplier_rating: Decimal 0.00-5.00, 2 decimals
        if i % 37 == 0:
            # String values (should be mapped)
            rating = random.choice(
                ["Excellent", "Good", "Fair", "Poor", "5 stars"]
            )
        elif i % 47 == 0:
            # Negative (out of range)
            rating = round(random.uniform(-1.0, -0.1), 2)
        elif i % 57 == 0:
            # Too high (> 5.00)
            rating = round(random.uniform(5.1, 10.0), 2)
        elif i % 67 == 0:
            # Too many decimals
            rating = round(random.uniform(0, 5), 5)
        elif i % 77 == 0:
            # Integer format
            rating = random.randint(1, 5)
        else:
            # Valid: realistic rating distribution (skewed toward higher)
            rating = round(
                random.choices(
                    [
                        random.uniform(0, 2),
                        random.uniform(2, 3.5),
                        random.uniform(3.5, 5),
                    ],
                    weights=[10, 30, 60],
                    k=1,
                )[0],
                2,
            )
        record["supplier_rating"] = rating

        # supplier_status: Canonical set
        if i % 32 == 0:
            # Case inconsistencies
            status = random.choice(
                ["active", "ACTIVE", "inactive", "INACTIVE", "pending", "PENDING"]
            )
        elif i % 42 == 0:
            # Typos
            status = random.choice(["actve", "Inactiv", "Pendng", "Suspnded"])
        elif i % 62 == 0:
            # Synonyms that should be mapped
            status = random.choice(["On Hold", "Disabled", "Blocked", "Approved"])
        else:
            # Valid: canonical values
            status = random.choices(STATUS_CANONICAL, weights=[50, 20, 10, 15, 5], k=1)[
                0
            ]
        record["supplier_status"] = status

        # is_preferred: Boolean, typically TRUE implies is_verified=TRUE
        if i % 36 == 0:
            # Various boolean representations
            is_preferred = random.choice(
                ["Y", "N", "Yes", "No", "1", "0", "true", "false"]
            )
        else:
            # Valid: 20% are preferred
            is_preferred = random.choices([True, False], weights=[20, 80], k=1)[0]
        record["is_preferred"] = is_preferred

        # is_verified: Boolean
        if i % 39 == 0:
            # Various boolean representations
            is_verified = random.choice(
                ["Y", "N", "Yes", "No", "1", "0", "true", "false"]
            )
        elif i % 59 == 0:
            # is_preferred=TRUE but is_verified=FALSE violation
            if is_preferred == True:
                is_verified = False
            else:
                is_verified = True
        elif i % 69 == 0:
            # Status=Suspended/Inactive but is_preferred=TRUE violation
            if status in ["Suspended", "Inactive"]:
                record["is_preferred"] = True
                is_verified = True
            else:
                is_verified = False
        else:
            # Valid: 70% are verified
            is_verified = random.choices([True, False], weights=[70, 30], k=1)[0]
        record["is_verified"] = is_verified

        # contract_start_date: YYYY-MM-DD
        if i % 33 == 0:
            # String format variations
            start_dt = fake.date_between(start_date="-5y", end_date="+6m")
            formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            contract_start = start_dt.strftime(random.choice(formats))
        elif i % 54 == 0:
            # Very old contracts (> 10 years)
            contract_start = fake.date_between(start_date="-15y", end_date="-11y")
        elif i % 64 == 0:
            # Far future contracts
            contract_start = fake.date_between(start_date="+2y", end_date="+5y")
        else:
            # Valid: within reasonable range
            contract_start = fake.date_between(start_date="-3y", end_date="+3m")
        record["contract_start_date"] = contract_start

        # contract_end_date: > contract_start_date, duration 1-5 years typical
        if i % 38 == 0:
            # End before start violation
            if isinstance(contract_start, date):
                contract_end = contract_start - timedelta(days=random.randint(1, 365))
            else:
                contract_end = fake.date_between(start_date="-6y", end_date="-5y")
        elif i % 48 == 0:
            # Same as start date (not strictly greater)
            contract_end = contract_start
        elif i % 58 == 0:
            # Very short contract (< 30 days)
            if isinstance(contract_start, date):
                contract_end = contract_start + timedelta(days=random.randint(1, 29))
            else:
                contract_end = fake.date_between(start_date="-1y", end_date="now")
        elif i % 68 == 0:
            # Very long contract (> 10 years)
            if isinstance(contract_start, date):
                contract_end = contract_start + timedelta(
                    days=random.randint(3700, 5500)
                )
            else:
                contract_end = fake.date_between(start_date="+10y", end_date="+15y")
        elif i % 78 == 0:
            # String format
            end_dt = fake.date_between(start_date="-1y", end_date="+5y")
            contract_end = end_dt.strftime("%m/%d/%Y")
        else:
            # Valid: 6 months to 5 years duration
            if isinstance(contract_start, date):
                duration_days = random.randint(180, 1825)
                contract_end = contract_start + timedelta(days=duration_days)
            else:
                contract_end = fake.date_between(start_date="-2y", end_date="+3y")
        record["contract_end_date"] = contract_end

        # supplier_name: Mandatory, 2-150 chars
        if i % 31 == 0:
            # Duplicates with variations
            if duplicate_names:
                base_name = random.choice(duplicate_names)
                variations = [
                    base_name,
                    base_name.upper(),
                    base_name.lower(),
                    base_name + " Inc",
                    base_name + " LLC",
                    base_name + " Co.",
                ]
                name = random.choice(variations)
            else:
                name = fake.company()
                duplicate_names.append(name)
        elif i % 51 == 0:
            # Too short (< 2 chars)
            name = random.choice(["A", "X", "Z"])
        elif i % 71 == 0:
            # Only numbers/symbols
            name = random.choice(["12345", "###", "@@@", "---"])
        elif i % 81 == 0:
            # Names with special characters
            name = fake.company() + random.choice(
                [" & Co.", " @ Supply", " #1", " *Premium*"]
            )
        else:
            # Valid company name
            name = fake.company()
        record["supplier_name"] = name

        # contact_email: Valid email format
        if i % 34 == 0:
            # Invalid formats
            invalid_emails = [
                "not-an-email",
                "@company.com",
                fake.user_name(),
                fake.user_name() + "@",
            ]
            email = random.choice(invalid_emails)
        elif i % 44 == 0:
            # Dummy domains
            email = f"{fake.user_name()}@example.com"
        elif i % 54 == 0:
            # Case issues
            email = fake.company_email().upper()
        else:
            # Valid email
            email = fake.company_email()
        record["contact_email"] = email

        # phone_number: E.164 format, 7-15 digits
        if i % 35 == 0:
            # Invalid formats
            invalid_phones = [
                "0000000000",
                "9999999999",
                "123",
                "555-555-5555",
                "+10000000000",
                "1111111111",
            ]
            phone = random.choice(invalid_phones)
        elif i % 45 == 0:
            # Too short (< 7 digits)
            phone = str(random.randint(100, 999999))
        elif i % 55 == 0:
            # Too long (> 15 digits)
            phone = str(random.randint(10**16, 10**18))
        else:
            # Valid: various formats (will need normalization)
            phone = fake.phone_number()
        record["phone_number"] = phone

        # tax_id: Country-specific format, unique
        if i % 56 == 0 and used_tax_ids:
            # Duplicate tax_id violation
            tax_id = random.choice(used_tax_ids)
        else:
            # Valid: country-specific formats
            tax_formats = [
                f"{random.randint(10, 99)}-{random.randint(1000000, 9999999)}",  # US EIN
                f"GB{random.randint(100000000, 999999999)}",  # UK VAT
                f"DE{random.randint(100000000, 999999999)}",  # DE VAT
                f"FR{random.randint(10000000000, 99999999999)}",  # FR VAT
            ]
            tax_id = random.choice(tax_formats)
            used_tax_ids.append(tax_id)
        record["tax_id"] = tax_id

        # city: 2-60 chars, no digits-only
        if i % 40 == 0:
            # Case inconsistencies
            city = random.choice([fake.city().upper(), fake.city().lower()])
        elif i % 50 == 0:
            # Special characters
            city = fake.city() + random.choice([" (Main)", " - HQ", " *", " #1"])
        elif i % 70 == 0:
            # Digits only (invalid)
            city = str(random.randint(10000, 99999))
        else:
            # Valid city
            city = fake.city()
        record["city"] = city

        # state: Valid subdivision for country
        if i % 35 == 0:
            # Mixed formats (abbrev vs full)
            state = random.choice([fake.state(), fake.state_abbr()])
        elif i % 55 == 0:
            # Case issues
            state = random.choice([fake.state().upper(), fake.state().lower()])
        else:
            # Valid state
            state = fake.state()
        record["state"] = state

        # zip_code: Country-specific format
        if i % 30 == 0:
            # Placeholder values
            zip_code = random.choice(
                ["00000", "99999", "XXXXX", "11111"]
            )
        elif i % 40 == 0:
            # International formats
            intl_formats = [
                f"{fake.country_code()}-{random.randint(1000, 9999)}",
                f"{random.choice(['SW', 'NW', 'SE', 'NE'])}{random.randint(1, 9)} {random.randint(1, 9)}{random.choice(['AA', 'BB'])}",
            ]
            zip_code = random.choice(intl_formats)
        elif i % 60 == 0:
            # Too short/long
            zip_code = (
                str(random.randint(100, 999))
                if random.random() > 0.5
                else str(random.randint(100000, 9999999))
            )
        else:
            # Valid US ZIP
            zip_code = fake.zipcode()
        record["zip_code"] = zip_code

        # country: ISO 3166-1 alpha-2
        if i % 36 == 0:
            # Country codes (valid ISO)
            country = random.choice(COUNTRY_CODES)
        elif i % 46 == 0:
            # ISO3 codes (should be alpha-2)
            iso3_codes = ["USA", "GBR", "DEU", "FRA", "CHN", "JPN"]
            country = random.choice(iso3_codes)
        elif i % 56 == 0:
            # Variations (should be normalized)
            country = random.choice(
                [
                    "United States",
                    "United States of America",
                    "USA",
                    "US",
                    "U.S.",
                    "U.S.A.",
                    "America",
                ]
            )
        elif i % 66 == 0:
            # Invalid/old country names
            old_countries = [
                "USSR",
                "Yugoslavia",
                "Czechoslovakia",
                "East Germany",
                "Burma",
            ]
            country = random.choice(old_countries)
        elif i % 86 == 0:
            # UK vs GB
            country = "UK"  # Should be GB for strict ISO
        else:
            # Valid: full country name
            country = fake.country()
        record["country"] = country

        # created_at: <= now(), <= contract_start_date
        if i % 41 == 0:
            # String timestamp
            created = fake.date_time_between(start_date="-5y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 63 == 0:
            # Future date violation
            created = fake.date_time_between(start_date="+1d", end_date="+1y")
        elif i % 73 == 0:
            # created_at > contract_start_date violation
            if isinstance(contract_start, date):
                created = datetime.combine(
                    contract_start + timedelta(days=random.randint(30, 365)),
                    datetime.min.time(),
                )
            else:
                created = fake.date_time_between(start_date="now", end_date="+1y")
        else:
            # Valid: before contract start
            if isinstance(contract_start, date):
                max_date = datetime.combine(contract_start, datetime.min.time())
                created = fake.date_time_between(start_date="-5y", end_date=max_date)
            else:
                created = fake.date_time_between(start_date="-5y", end_date="now")
        record["created_at"] = created

        data.append(record)

    df = pd.DataFrame(data)

    # Add exact duplicates (2%)
    for _ in range(int(num_rows * 0.02)):
        if len(df) > 0:
            df = pd.concat([df, df.sample(1)], ignore_index=True)

    df = df.sample(frac=1).reset_index(drop=True)
    return df


def add_supplier_messiness(df):
    string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    for col in string_cols[:3]:
        mask = np.random.random(len(df)) < 0.02
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


df = generate_messy_supplier_data(number_of_suppliers)
df = add_supplier_messiness(df)

output_file = "suppliers.xlsx"
df.to_excel(output_file, index=False)